# Bronze → Silver — Faturas

Base de código: arquivo 'Landing_to_Bronze.ipynb' do PR#14. 

Reconstrução incremental da Silver, começando por faturas. O guia CineData é uma referência de método; suas regras e resultados de filmes não são regras do V-Energy.

## Etapa 1 — conhecer a origem

Objetivo: identificar a tabela, fixar uma versão Delta para a exploração e observar schema, volume e exemplos brutos.

Nesta etapa só há consultas. Nenhuma tabela é criada ou sobrescrita e nenhum registro é limpo, convertido, filtrado ou deduplicado.

A contagem e a amostra descrevem a origem; não comprovam qualidade. A amostra não representa necessariamente todos os formatos existentes.


In [0]:
# configuração
CATALOGO = "v-energy"
SCHEMA_BRONZE = "bronze"
TABELA_FATURAS = "tb_faturas"

bronze_faturas = f"`{CATALOGO}`.`{SCHEMA_BRONZE}`.`{TABELA_FATURAS}`"
print(f"Origem: {bronze_faturas}")


Origem: `v-energy`.`bronze`.`tb_faturas`


## Célula 2 — versão, estrutura e amostra

A Bronze dessa implementação recebe a extração completa por overwrite. Por isso, não vamos selecionar automaticamente apenas o maior ingestion_datetime nem remover supostos batches duplicados.

Fixar a versão Delta permite que as próximas consultas usem o mesmo estado mesmo se a Bronze for atualizada durante a exploração. Reexecutar esta célula seleciona a versão mais recente naquele momento.

Observe: versão e operação da origem, nomes e tipos de colunas, quantidade total e exemplos de datas, moedas, status e metadados. Os valores ainda são brutos, incluindo eventuais cabeçalhos internos e sentinelas.


In [0]:
# exploração inicial, sem transformar ou gravar os dados.
historico_origem = spark.sql(
    f"DESCRIBE HISTORY {bronze_faturas} LIMIT 1"
).select("version", "timestamp", "operation").first()

VERSAO_BRONZE_FATURAS = historico_origem["version"]
df_faturas_bronze = (
    spark.read
    .option("versionAsOf", VERSAO_BRONZE_FATURAS)
    .table(bronze_faturas)
)

print(f"Versão Delta: {VERSAO_BRONZE_FATURAS}")
print(f"Timestamp do commit Delta: {historico_origem['timestamp']}")
print(f"Operação Delta: {historico_origem['operation']}")
print(f"Total de linhas na versão: {df_faturas_bronze.count():,}")
df_faturas_bronze.printSchema()
display(df_faturas_bronze.limit(10))


Versão Delta: 0
Timestamp do commit Delta: 2026-10-22 18:39:25
Operação Delta: CREATE OR REPLACE TABLE AS SELECT
Total de linhas na versão: 2,184,623
root
 |-- id_fatura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_emissao: string (nullable = true)
 |-- valor_energia: string (nullable = true)
 |-- tributos: string (nullable = true)
 |-- bandeira_tarifaria: string (nullable = true)
 |-- valor_total: string (nullable = true)
 |-- data_vencimento: string (nullable = true)
 |-- data_pagamento: string (nullable = true)
 |-- status: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_datetime
FT000000000,UC100000,2022-07,2022-07-27,76.93,25.64,Verde,102.57,2022-08-08,2022-08-23,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000001,UC100000,2022-08,2022-08-11,86.70,28.90,Verde,115.60,2022-08-23,2022-08-30,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000002,UC100000,2022-09,2022-09-12,97.43,32.48,Verde,129.91,2022-09-24,2022-09-30,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000003,UC100000,2022-10,2022-10-28,125.95,41.98,Verde,167.93,2022-11-09,2022-11-07,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000004,UC100000,2022-11,2022-11-24,175.64,58.55,Verde,234.19,2022-12-06,null,EM ABERTO,faturas.csv,2026-10-22T18:39:16.718Z
FT000000005,UC100000,2022-12,2022-12-05,232.26,77.42,Verde,309.68,2022-12-17,2022-12-23,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000006,UC100000,2023-01,2023-01-19,190.32,63.44,Verde,253.76,2023-01-31,2023-02-21,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000007,UC100000,2023-02,2023-02-06,223.55,74.52,Verde,298.07,2023-02-18,2023-02-24,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000008,UC100000,2023-03,2023-03-10,192.42,64.14,Verde,256.56,2023-03-22,2023-04-06,PAGA,faturas.csv,2026-10-22T18:39:16.718Z
FT000000009,UC100000,2023-04,2023-04-30,191.71,63.90,Verde,255.61,2023-05-12,2023-05-31,PAGA,faturas.csv,2026-10-22T18:39:16.718Z


## Próxima etapa — diagnóstico orientado por evidências

Depois de observar as saídas, medir cabeçalhos internos, nulos, campos vazios, espaços, domínios e padrões. Em seguida, analisar duplicatas exatas, repetição de id_fatura e multiplicidade UC/competência separadamente.

Cada tratamento será registrado como: evidência observada → alternativas → decisão e justificativa → implementação → validação e impacto.

Ainda não foi escolhida regra de deduplicação, interpretação de datas ambíguas ou destino de registros sem referência cadastral. O snapshot anterior e sua proposta de quarentena são material histórico, não o contrato deste notebook.


## 2. Diagnóstico de qualidade da Bronze de faturas

Objetivo: medir os problemas existentes antes de escolher tratamentos.

As consultas utilizam a versão Delta fixada na seção anterior.

Nesta seção:
- identificamos cabeçalhos internos pelo conjunto completo dos onze campos;
- medimos nulos, vazios e espaços sem normalizar os dados;
- distinguimos duplicatas exatas, repetição de ID e multiplicidade por UC/mês;
- observamos os domínios brutos de status e bandeira.

Nenhuma tabela é alterada e nenhuma deduplicação é aplicada.

In [0]:
from pyspark.sql import functions as F

COLUNAS_NEGOCIO = [
    "id_fatura",
    "id_uc",
    "mes_referencia",
    "data_emissao",
    "valor_energia",
    "tributos",
    "bandeira_tarifaria",
    "valor_total",
    "data_vencimento",
    "data_pagamento",
    "status",
]

colunas_ausentes = set(COLUNAS_NEGOCIO) - set(df_faturas_bronze.columns)

if colunas_ausentes:
    raise ValueError(f"Colunas de negócio ausentes: {sorted(colunas_ausentes)}")

eh_cabecalho = F.lit(True)

for coluna in COLUNAS_NEGOCIO:
    eh_cabecalho = (
        eh_cabecalho
        & F.col(coluna).eqNullSafe(F.lit(coluna))
    )

df_diagnostico = df_faturas_bronze.withColumn(
    "_eh_cabecalho_interno",
    eh_cabecalho,
)

display(
    df_diagnostico
    .groupBy("_eh_cabecalho_interno")
    .count()
    .withColumnRenamed("count", "quantidade")
)

# Mantemos os cabeçalhos disponíveis para inspeção.
df_cabecalhos_internos = (
    df_diagnostico
    .filter(F.col("_eh_cabecalho_interno"))
    .drop("_eh_cabecalho_interno")
)

# Esta visão será usada nos diagnósticos de conteúdo.
# Nenhuma linha é apagada da Bronze.
df_faturas_analise = (
    df_diagnostico
    .filter(~F.col("_eh_cabecalho_interno"))
    .drop("_eh_cabecalho_interno")
)

display(df_cabecalhos_internos)

## Decisão nesta etapa: analisar cabeçalhos separadamente, porque eles não representam faturas. A auditoria anterior registrava dois, mas vamos conferir o resultado atual sem impor essa quantidade ao código.

_eh_cabecalho_interno,quantidade
false,2184621
true,2


id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,source_file,ingestion_datetime
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,faturas.csv,2026-10-22T18:39:16.718Z
id_fatura,id_uc,mes_referencia,data_emissao,valor_energia,tributos,bandeira_tarifaria,valor_total,data_vencimento,data_pagamento,status,faturas.csv,2026-10-22T18:39:16.718Z


In [0]:
# separar situações diferentes: nulo real, texto vazio ou só com espaços, espaços nas extremidades e texto literal "null"
expressoes = []

for coluna in COLUNAS_NEGOCIO:
    valor = F.col(coluna)
    texto = F.trim(valor)

    expressoes.extend([
        F.count(
            F.when(valor.isNull(), 1)
        ).alias(f"{coluna}__nulos"),

        F.count(
            F.when(valor.isNotNull() & (texto == ""), 1)
        ).alias(f"{coluna}__vazios"),

        F.count(
            F.when(valor != texto, 1)
        ).alias(f"{coluna}__espacos"),

        F.count(
            F.when(F.lower(texto) == "null", 1)
        ).alias(f"{coluna}__texto_null"),
    ])

contagens = df_faturas_analise.agg(*expressoes).first().asDict()

perfil_ausencias = [
    (
        coluna,
        contagens[f"{coluna}__nulos"],
        contagens[f"{coluna}__vazios"],
        contagens[f"{coluna}__espacos"],
        contagens[f"{coluna}__texto_null"],
    )
    for coluna in COLUNAS_NEGOCIO
]

display(
    spark.createDataFrame(
        perfil_ausencias,
        """
        coluna string,
        nulos_reais long,
        vazios_ou_so_espacos long,
        com_espacos_nas_extremidades long,
        texto_literal_null long
        """,
    )
)

# Como interpretar: nulos em data_pagamento podem ser compatíveis com faturas em aberto. Já um identificador nulo exige outra avaliação. Portanto, não vamos preencher todos os nulos com zero ou aplicar uma regra única de descarte.

# As contagens de espaços podem incluir textos compostos apenas por espaços; essas categorias não devem ser somadas como registros distintos com problema.

coluna,nulos_reais,vazios_ou_so_espacos,com_espacos_nas_extremidades,texto_literal_null
id_fatura,0,0,0,0
id_uc,0,0,0,0
mes_referencia,0,0,0,0
data_emissao,0,0,0,0
valor_energia,0,0,0,0
tributos,0,0,0,0
bandeira_tarifaria,0,0,0,0
valor_total,0,0,0,0
data_vencimento,0,0,0,0
data_pagamento,113835,0,0,0


In [0]:
# perguntas: 
# 1 - Existem cópias exatas do mesmo registro de negócio?
# 2 - O mesmo identificador aparece em mais de uma linha?
# 3 - Há mais de uma fatura para a mesma UC e competência?

def resumir_repeticoes(colunas, criterio):
    grupos_repetidos = (
        df_faturas_analise
        .groupBy(*colunas)
        .agg(F.count("*").alias("quantidade"))
        .filter(F.col("quantidade") > 1)
    )

    return (
        grupos_repetidos
        .agg(
            F.count("*").alias("grupos_repetidos"),

            F.coalesce(
                F.sum("quantidade"),
                F.lit(0),
            ).alias("linhas_nesses_grupos"),

            F.coalesce(
                F.sum(F.col("quantidade") - 1),
                F.lit(0),
            ).alias("linhas_alem_de_uma_por_grupo"),
        )
        .withColumn("criterio", F.lit(criterio))
        .select(
            "criterio",
            "grupos_repetidos",
            "linhas_nesses_grupos",
            "linhas_alem_de_uma_por_grupo",
        )
    )


resumo_repeticoes = (
    resumir_repeticoes(
        COLUNAS_NEGOCIO,
        "Registro de negócio completo",
    )
    .unionByName(
        resumir_repeticoes(["id_fatura"], "ID da fatura")
    )
    .unionByName(
        resumir_repeticoes(
            ["id_uc", "mes_referencia"],
            "UC e competência",
        )
    )
)

display(resumo_repeticoes)

# Ainda não há decisão de deduplicação. Duas faturas com IDs distintos para a mesma UC/mês não são necessariamente cópias. A coluna linhas_alem_de_uma_por_grupo é uma medida de multiplicidade, não uma recomendação de exclusão.

# Os metadados ficaram fora da comparação de duplicatas exatas para não esconder cópias de negócio que tenham carimbos de ingestão diferentes. Nesta primeira análise, os textos são comparados como chegaram; possíveis colisões após normalização serão verificadas depois.

criterio,grupos_repetidos,linhas_nesses_grupos,linhas_alem_de_uma_por_grupo
Registro de negócio completo,0,0,0
ID da fatura,0,0,0
UC e competência,21,42,21


In [0]:
for coluna in ["status", "bandeira_tarifaria"]:
    print(f"Distribuição bruta de: {coluna}")

    display(
        df_faturas_analise
        .groupBy(coluna)
        .agg(F.count("*").alias("quantidade"))
        .orderBy(F.desc("quantidade"))
    )

    # Decisão a avaliar: se houver diferenças apenas de caixa ou espaços, poderemos propor uma padronização. Valores desconhecidos precisam ser investigados antes de receber um mapeamento.

Distribuição bruta de: status


status,quantidade
PAGA,2009453
EM ABERTO,175168


Distribuição bruta de: bandeira_tarifaria


bandeira_tarifaria,quantidade
Verde,1546130
Amarela,273630
Vermelha P1,228042
Vermelha P2,136819


Os status são somente PAGA e EM ABERTO. As bandeiras apresentam quatro categorias: Verde, Amarela, Vermelha P1 e Vermelha P2.
Duas decisões já têm fundamento:
- Separar os dois cabeçalhos internos: são registros estruturais, preservados na Bronze e disponíveis em df_cabecalhos_internos.
- Não aplicar deduplicação neste momento: não existem cópias exatas nem IDs repetidos. As 42 faturas precisam de investigação própria; ter a mesma UC e competência não prova duplicação.
Também não há necessidade demonstrada de aplicar trim ou mudar a capitalização. Podemos padronizar a apresentação depois, mas isso seria uma escolha de contrato.

Também não há necessidade demonstrada de aplicar trim ou mudar a capitalização. Podemos padronizar a apresentação depois, mas isso seria uma escolha de contrato.

## 3. Diagnóstico de pagamentos

Objetivos:
- distinguir pagamento nulo, sentinela, data válida e conteúdo inválido;
- cruzar essa classificação com o status;
- verificar o formato e o calendário do vencimento;
- medir pagamentos posteriores ao vencimento.

As colunas originais permanecem intactas. As conversões são auxiliares
de diagnóstico, identificadas pelo prefixo "_diag_".

Um pagamento ausente não será preenchido com zero, vencimento ou data atual.
A ausência precisa ser interpretada em conjunto com o status.

Pagamento posterior ao vencimento representa atraso, não necessariamente
erro de qualidade.

Existem 175.168 faturas EM ABERTO, mas somente 113.835 pagamentos nulos. Isso ainda não demonstra inconsistência: precisamos descobrir o conteúdo do campo nas demais.
Vamos investigar pagamentos antes de escolher qualquer preenchimento ou conversão definitiva.

A sentinela 9999-12-31, registrada na auditoria anterior, será contada separadamente. Vamos verificar sua presença nesta execução.
As datas auxiliares abaixo só aceitam o padrão ISO AAAA-MM-DD com calendário válido. Outro formato será identificado para investigação.

In [0]:
def interpretar_data_iso(coluna):
    texto = F.col(coluna)

    return F.when(
        texto.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        F.to_date(
            F.try_to_timestamp(texto, F.lit("yyyy-MM-dd"))
        ),
    )


df_pagamentos_diagnostico = (
    df_faturas_analise
    .withColumn(
        "_diag_pagamento",
        F.when(
            F.col("data_pagamento") != "9999-12-31",
            interpretar_data_iso("data_pagamento"),
        ),
    )
    .withColumn(
        "_diag_vencimento",
        interpretar_data_iso("data_vencimento"),
    )
    .withColumn(
        "_diag_tipo_pagamento",
        F.when(
            F.col("data_pagamento").isNull(),
            "NULO_REAL",
        )
        .when(
            F.trim("data_pagamento") == "",
            "VAZIO_OU_ESPACOS",
        )
        .when(
            F.col("data_pagamento") == "9999-12-31",
            "SENTINELA_9999_12_31",
        )
        .when(
            ~F.col("data_pagamento").rlike(r"^\d{4}-\d{2}-\d{2}$"),
            "OUTRO_FORMATO",
        )
        .when(
            F.col("_diag_pagamento").isNull(),
            "DATA_ISO_INVALIDA",
        )
        .otherwise("DATA_ISO_VALIDA"),
    )
)

display(
    df_pagamentos_diagnostico
    .groupBy("status", "_diag_tipo_pagamento")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("status", "_diag_tipo_pagamento")
)

status,_diag_tipo_pagamento,quantidade
EM ABERTO,NULO_REAL,113835
EM ABERTO,SENTINELA_9999_12_31,61333
PAGA,DATA_ISO_VALIDA,2009453


O que observar: se nulos e sentinelas aparecem exclusivamente em faturas EM ABERTO, e se todas as PAGA apresentam uma data válida. Se surgir outro formato, precisaremos examiná-lo antes de considerá-lo inválido.

In [0]:
regras_diagnostico = {
    "paga_sem_data_valida": (
        (F.col("status") == "PAGA")
        & F.col("_diag_pagamento").isNull()
    ),

    "em_aberto_com_data_valida": (
        (F.col("status") == "EM ABERTO")
        & F.col("_diag_pagamento").isNotNull()
    ),

    "pagamento_com_outro_formato": (
        F.col("_diag_tipo_pagamento") == "OUTRO_FORMATO"
    ),

    "pagamento_com_calendario_invalido": (
        F.col("_diag_tipo_pagamento") == "DATA_ISO_INVALIDA"
    ),

    "vencimento_ausente_ou_nao_interpretado": (
        F.col("_diag_vencimento").isNull()
    ),

    "pagamento_apos_vencimento": (
        F.col("_diag_pagamento") > F.col("_diag_vencimento")
    ),
}

display(
    df_pagamentos_diagnostico.agg(*[
        F.count(
            F.when(condicao, 1)
        ).alias(nome)
        for nome, condicao in regras_diagnostico.items()
    ])
)

paga_sem_data_valida,em_aberto_com_data_valida,pagamento_com_outro_formato,pagamento_com_calendario_invalido,vencimento_ausente_ou_nao_interpretado,pagamento_apos_vencimento
0,0,0,0,0,1675636


Decisão a avaliar: se a matriz confirmar que nulo e sentinela representam ausência de pagamento em faturas abertas, poderemos propor uma coluna tipada com NULL nesses casos, preservando o motivo e o valor original.

O indicador de atraso considera somente comparações com pagamento e vencimento interpretados. Ele não mede há quanto tempo uma fatura em aberto está vencida.

In [0]:
# Intervalos das datas interpretadas.
display(
    df_pagamentos_diagnostico.agg(
        F.min("_diag_vencimento").alias("menor_vencimento"),
        F.max("_diag_vencimento").alias("maior_vencimento"),
        F.min("_diag_pagamento").alias("menor_pagamento"),
        F.max("_diag_pagamento").alias("maior_pagamento"),
    )
)

# Atraso não entra neste filtro: ele é analisado separadamente.
exige_investigacao = (
    regras_diagnostico["paga_sem_data_valida"]
    | regras_diagnostico["em_aberto_com_data_valida"]
    | regras_diagnostico["pagamento_com_outro_formato"]
    | regras_diagnostico["pagamento_com_calendario_invalido"]
    | regras_diagnostico["vencimento_ausente_ou_nao_interpretado"]
)

display(
    df_pagamentos_diagnostico
    .filter(exige_investigacao)
    .select(
        "id_fatura",
        "status",
        "data_vencimento",
        "data_pagamento",
        "_diag_tipo_pagamento",
        "_diag_vencimento",
        "_diag_pagamento",
    )
    .limit(20)
)

menor_vencimento,maior_vencimento,menor_pagamento,maior_pagamento
2022-07-15,2026-07-12,2022-07-11,2026-08-06


id_fatura,status,data_vencimento,data_pagamento,_diag_tipo_pagamento,_diag_vencimento,_diag_pagamento


## 4. Diagnóstico da data de emissão

Objetivos:
- identificar os formatos existentes;
- verificar se as máscaras produzem datas de calendário válidas;
- distinguir interpretação única de ambiguidade;
- observar exemplos antes de escolher uma convenção.

Máscaras testadas:
- ano-mês-dia para YYYY-MM-DD;
- dia/mês/ano para DD/MM/YYYY;
- dia.mês.ano para DD.MM.YYYY;
- mês-dia-ano e dia-mês-ano para datas com hífen e ano no fim.

DMY para barras e pontos é uma hipótese de interpretação nesta exploração,
ainda sujeita à avaliação. Um parsing bem-sucedido comprova validade de
calendário segundo a máscara, não o significado verdadeiro da origem.

Nenhuma data original é alterada e nenhuma interpretação ambígua é escolhida.

In [0]:
# Esta classificação reconhece a aparência do texto. Uma data como 2025-02-30 teria aparência ISO, mas só será identificada como calendário inválido na próxima célula.

emissao = F.col("data_emissao")

df_emissao_diagnostico = df_faturas_analise.withColumn(
    "_diag_formato_emissao",
    F.when(emissao.isNull(), "NULO")
    .when(F.trim(emissao) == "", "VAZIO")
    .when(
        emissao.rlike(r"^\d{4}-\d{2}-\d{2}$"),
        "ISO_ANO_MES_DIA",
    )
    .when(
        emissao.rlike(r"^\d{2}/\d{2}/\d{4}$"),
        "BARRAS_ANO_NO_FIM",
    )
    .when(
        emissao.rlike(r"^\d{2}\.\d{2}\.\d{4}$"),
        "PONTOS_ANO_NO_FIM",
    )
    .when(
        emissao.rlike(r"^\d{2}-\d{2}-\d{4}$"),
        "HIFEN_ANO_NO_FIM",
    )
    .otherwise("OUTRO_FORMATO"),
)

display(
    df_emissao_diagnostico
    .groupBy("_diag_formato_emissao")
    .agg(F.count("*").alias("quantidade"))
    .orderBy(F.desc("quantidade"))
)

_diag_formato_emissao,quantidade
ISO_ANO_MES_DIA,2001260
BARRAS_ANO_NO_FIM,61291
PONTOS_ANO_NO_FIM,61252
HIFEN_ANO_NO_FIM,60818


In [0]:
# As colunas auxiliares guardarão as datas candidatas. Para hífen com ano no fim, testaremos as duas ordens, sem usar a competência para escolher uma delas.

def testar_mascara_emissao(formato, mascara):
    return F.when(
        F.col("_diag_formato_emissao") == formato,
        F.to_date(
            F.try_to_timestamp(
                F.col("data_emissao"),
                F.lit(mascara),
            )
        ),
    )


df_emissao_diagnostico = (
    df_emissao_diagnostico
    .withColumn(
        "_diag_emissao_por_mascara",
        F.coalesce(
            testar_mascara_emissao(
                "ISO_ANO_MES_DIA", "yyyy-MM-dd"
            ),
            testar_mascara_emissao(
                "BARRAS_ANO_NO_FIM", "dd/MM/yyyy"
            ),
            testar_mascara_emissao(
                "PONTOS_ANO_NO_FIM", "dd.MM.yyyy"
            ),
        ),
    )
    .withColumn(
        "_diag_emissao_mdy",
        testar_mascara_emissao(
            "HIFEN_ANO_NO_FIM", "MM-dd-yyyy"
        ),
    )
    .withColumn(
        "_diag_emissao_dmy",
        testar_mascara_emissao(
            "HIFEN_ANO_NO_FIM", "dd-MM-yyyy"
        ),
    )
    .withColumn(
        "_diag_emissao_ambigua",
        F.coalesce(
            F.col("_diag_emissao_mdy")
            != F.col("_diag_emissao_dmy"),
            F.lit(False),
        ),
    )
)

tem_alguma_data = F.coalesce(
    F.col("_diag_emissao_por_mascara"),
    F.col("_diag_emissao_mdy"),
    F.col("_diag_emissao_dmy"),
).isNotNull()

df_emissao_diagnostico = df_emissao_diagnostico.withColumn(
    "_diag_interpretacao_emissao",
    F.when(
        ~tem_alguma_data,
        "SEM_INTERPRETACAO_VALIDA",
    )
    .when(
        F.col("_diag_emissao_ambigua"),
        "DUAS_DATAS_DISTINTAS",
    )
    .when(
        F.col("_diag_emissao_mdy").isNotNull()
        & F.col("_diag_emissao_dmy").isNotNull(),
        "MDY_E_DMY_COINCIDENTES",
    )
    .when(
        F.col("_diag_emissao_mdy").isNotNull(),
        "SOMENTE_MDY_VALIDO",
    )
    .when(
        F.col("_diag_emissao_dmy").isNotNull(),
        "SOMENTE_DMY_VALIDO",
    )
    .otherwise("VALIDA_PELA_MASCARA_TESTADA"),
)

display(
    df_emissao_diagnostico
    .groupBy(
        "_diag_formato_emissao",
        "_diag_interpretacao_emissao",
    )
    .agg(F.count("*").alias("quantidade"))
    .orderBy(
        "_diag_formato_emissao",
        "_diag_interpretacao_emissao",
    )
)

_diag_formato_emissao,_diag_interpretacao_emissao,quantidade
BARRAS_ANO_NO_FIM,VALIDA_PELA_MASCARA_TESTADA,61291
HIFEN_ANO_NO_FIM,DUAS_DATAS_DISTINTAS,20416
HIFEN_ANO_NO_FIM,MDY_E_DMY_COINCIDENTES,1787
HIFEN_ANO_NO_FIM,SOMENTE_MDY_VALIDO,38615
ISO_ANO_MES_DIA,VALIDA_PELA_MASCARA_TESTADA,2001260
PONTOS_ANO_NO_FIM,VALIDA_PELA_MASCARA_TESTADA,61252


Como interpretar:
- Somente MDY ou DMY válido: apenas uma das duas máscaras produziu uma data.
- MDY e DMY coincidentes: ambas produziram a mesma data, como 03-03-2025.
- Duas datas distintas: a representação continua ambígua.
- Sem interpretação válida: nenhuma máscara testada interpretou o valor; precisamos inspecioná-lo.

Uma predominância de MDY será evidência para discutir uma convenção, sem provar individualmente o significado de todas as datas ambíguas.

In [0]:
# A competência será validada como AAAA-MM. Ainda não vamos exigir que a emissão ocorra nesse mesmo mês: essa relação precisa ser analisada separadamente.

competencia_valida = F.col("mes_referencia").rlike(
    r"^[1-9][0-9]{3}-(0[1-9]|1[0-2])$"
)

display(
    df_emissao_diagnostico.agg(
        F.count("*").alias("registros_analisados"),

        F.count(
            F.when(
                F.col("mes_referencia").isNull()
                | ~competencia_valida,
                1,
            )
        ).alias("competencias_invalidas"),

        F.count(
            F.when(
                F.col("_diag_emissao_ambigua"),
                1,
            )
        ).alias("emissoes_ambiguas"),

        F.count(
            F.when(
                F.col("_diag_interpretacao_emissao")
                == "SEM_INTERPRETACAO_VALIDA",
                1,
            )
        ).alias("emissoes_sem_interpretacao"),
    )
)

for classificacao in [
    "DUAS_DATAS_DISTINTAS",
    "SEM_INTERPRETACAO_VALIDA",
]:
    print(f"Exemplos: {classificacao}")

    display(
        df_emissao_diagnostico
        .filter(
            F.col("_diag_interpretacao_emissao")
            == classificacao
        )
        .select(
            "id_fatura",
            "mes_referencia",
            "data_emissao",
            "_diag_formato_emissao",
            "_diag_emissao_por_mascara",
            "_diag_emissao_mdy",
            "_diag_emissao_dmy",
            "data_vencimento",
            "data_pagamento",
        )
        .limit(10)
    )

registros_analisados,competencias_invalidas,emissoes_ambiguas,emissoes_sem_interpretacao
2184621,0,20416,0


Exemplos: DUAS_DATAS_DISTINTAS


id_fatura,mes_referencia,data_emissao,_diag_formato_emissao,_diag_emissao_por_mascara,_diag_emissao_mdy,_diag_emissao_dmy,data_vencimento,data_pagamento
FT000000056,2023-03,03-10-2023,HIFEN_ANO_NO_FIM,null,2023-03-10,2023-10-03,2023-03-22,2023-04-13
FT000000143,2026-06,06-04-2026,HIFEN_ANO_NO_FIM,null,2026-06-04,2026-04-06,2026-06-16,2026-06-15
FT000000232,2025-11,11-08-2025,HIFEN_ANO_NO_FIM,null,2025-11-08,2025-08-11,2025-11-20,2025-11-28
FT000000348,2023-07,07-10-2023,HIFEN_ANO_NO_FIM,null,2023-07-10,2023-10-07,2023-07-22,2023-07-30
FT000000357,2024-04,04-05-2024,HIFEN_ANO_NO_FIM,null,2024-04-05,2024-05-04,2024-04-17,2024-04-23
FT000000418,2025-05,05-04-2025,HIFEN_ANO_NO_FIM,null,2025-05-04,2025-04-05,2025-05-16,2025-05-13
FT000000420,2025-07,07-05-2025,HIFEN_ANO_NO_FIM,null,2025-07-05,2025-05-07,2025-07-17,2025-08-03
FT000000628,2022-11,11-04-2022,HIFEN_ANO_NO_FIM,null,2022-11-04,2022-04-11,2022-11-16,2022-11-19
FT000000683,2023-06,06-07-2023,HIFEN_ANO_NO_FIM,null,2023-06-07,2023-07-06,2023-06-19,2023-06-19
FT000000691,2024-02,02-10-2024,HIFEN_ANO_NO_FIM,null,2024-02-10,2024-10-02,2024-02-22,2024-03-10


Exemplos: SEM_INTERPRETACAO_VALIDA


id_fatura,mes_referencia,data_emissao,_diag_formato_emissao,_diag_emissao_por_mascara,_diag_emissao_mdy,_diag_emissao_dmy,data_vencimento,data_pagamento


Não foram encontradas competências inválidas nem emissões sem interpretação pelas máscaras testadas.

A evidência favorece MDY para hífen com ano no fim: existem 38.615 casos exclusivamente MDY e nenhum exclusivamente DMY. Isso fundamenta uma possível convenção do projeto, mas ainda precisamos medir seus efeitos.

Os exemplos mostram por que a cronologia ajuda, mas nem sempre resolve:
- Em FT000000056, DMY produziria uma emissão em outubro, posterior ao vencimento de março.
- Em FT000000143, as duas candidatas são anteriores ao vencimento e ao pagamento. A cronologia permite ambas.

O ideal é não escolher automaticamente a data que coincide com a competência.

## 5. Comparação dos cenários de interpretação da emissão

Objetivos:
- medir quantas datas cada cenário consegue interpretar;
- comparar emissão com competência, vencimento e pagamento;
- identificar ambiguidades em que ambas, apenas uma ou nenhuma
  das candidatas respeitam a cronologia testada.

Condições cronológicas avaliadas:
- emissão não posterior ao vencimento;
- quando houver pagamento real, emissão não posterior ao pagamento.

Essas condições são critérios de diagnóstico a avaliar para o contrato
do projeto. Uma violação exige investigação, não alteração automática da data.

Coincidir com a competência é um indicador separado. Não será usado
para selecionar uma interpretação.

Nenhuma tabela é gravada e nenhuma convenção definitiva é aplicada.

Vamos comparar dois cenários: interpretar todos os valores com hífen e ano no fim como MDY ou como DMY. Para barras e pontos, manteremos a hipótese DMY usada na exploração anterior.

In [0]:
# As colunas originais continuam intactas. Os cenários criam representações auxiliares para comparação.

df_cronologia_base = (
    df_emissao_diagnostico
    .withColumn(
        "_diag_vencimento",
        interpretar_data_iso("data_vencimento"),
    )
    .withColumn(
        "_diag_pagamento",
        F.when(
            F.col("data_pagamento") != "9999-12-31",
            interpretar_data_iso("data_pagamento"),
        ),
    )
)

pagamento_ausente = (
    F.col("data_pagamento").isNull()
    | (F.col("data_pagamento") == "9999-12-31")
)


def cronologia_compativel(coluna_emissao):
    data = F.col(coluna_emissao)

    return F.coalesce(
        data.isNotNull()
        & F.col("_diag_vencimento").isNotNull()
        & (data <= F.col("_diag_vencimento"))
        & (
            pagamento_ausente
            | (
                F.col("_diag_pagamento").isNotNull()
                & (data <= F.col("_diag_pagamento"))
            )
        ),
        F.lit(False),
    )


def montar_cenario(nome, coluna_hifen):
    return (
        df_cronologia_base
        .withColumn("_diag_cenario", F.lit(nome))
        .withColumn(
            "_diag_emissao_cenario",
            F.when(
                F.col("_diag_formato_emissao")
                == "HIFEN_ANO_NO_FIM",
                F.col(coluna_hifen),
            ).otherwise(
                F.col("_diag_emissao_por_mascara")
            ),
        )
    )


df_cenarios_emissao = (
    montar_cenario("HIFEN_COMO_MDY", "_diag_emissao_mdy")
    .unionByName(
        montar_cenario("HIFEN_COMO_DMY", "_diag_emissao_dmy")
    )
    .withColumn(
        "_diag_cronologia_compativel",
        cronologia_compativel("_diag_emissao_cenario"),
    )
)



A compatibilidade exige que a mesma data candidata respeite vencimento e pagamento simultaneamente. Não basta uma interpretação passar em uma comparação e outra passar na comparação seguinte.

In [0]:
data_cenario = F.col("_diag_emissao_cenario")

display(
    df_cenarios_emissao
    .groupBy("_diag_cenario")
    .agg(
        F.count("*").alias("registros"),

        F.count(
            F.when(data_cenario.isNull(), 1)
        ).alias("emissoes_nao_interpretadas"),

        F.count(
            F.when(
                data_cenario.isNotNull()
                & (
                    F.date_format(data_cenario, "yyyy-MM")
                    != F.col("mes_referencia")
                ),
                1,
            )
        ).alias("emissoes_fora_competencia"),

        F.count(
            F.when(
                data_cenario > F.col("_diag_vencimento"),
                1,
            )
        ).alias("emissoes_apos_vencimento"),

        F.count(
            F.when(
                F.col("_diag_pagamento") < data_cenario,
                1,
            )
        ).alias("pagamentos_antes_emissao"),

        F.count(
            F.when(
                F.col("_diag_cronologia_compativel"),
                1,
            )
        ).alias("cronologia_compativel"),

        F.count(
            F.when(
                ~F.col("_diag_cronologia_compativel"),
                1,
            )
        ).alias("exigem_investigacao"),
    )
    .orderBy("_diag_cenario")
)

_diag_cenario,registros,emissoes_nao_interpretadas,emissoes_fora_competencia,emissoes_apos_vencimento,pagamentos_antes_emissao,cronologia_compativel,exigem_investigacao
HIFEN_COMO_DMY,2184621,38615,31269,11905,10464,2134101,50520
HIFEN_COMO_MDY,2184621,0,11306,0,0,2184621,0


Como interpretar:
- emissoes_nao_interpretadas: datas que a convenção daquele cenário não consegue interpretar.
- emissoes_fora_competencia: considera somente datas interpretadas; não representa automaticamente erro.
- exigem_investigacao: inclui ausência de interpretação e incompatibilidade cronológica.
- As categorias podem se sobrepor; não devem ser somadas como faturas distintas com problema.

In [0]:
# olhar apenas as 20.416 emissões com duas candidatas distintas.

df_ambiguidades_cronologia = (
    df_cronologia_base
    .filter(F.col("_diag_emissao_ambigua"))
    .withColumn(
        "_diag_mdy_compativel",
        cronologia_compativel("_diag_emissao_mdy"),
    )
    .withColumn(
        "_diag_dmy_compativel",
        cronologia_compativel("_diag_emissao_dmy"),
    )
)

display(
    df_ambiguidades_cronologia
    .groupBy(
        "_diag_mdy_compativel",
        "_diag_dmy_compativel",
    )
    .agg(F.count("*").alias("quantidade"))
    .orderBy(
        "_diag_mdy_compativel",
        "_diag_dmy_compativel",
    )
)

# Exemplos que exigiriam investigação no cenário MDY.
# Inclui todos os formatos, não apenas as emissões ambíguas.
display(
    df_cenarios_emissao
    .filter(
        (F.col("_diag_cenario") == "HIFEN_COMO_MDY")
        & ~F.col("_diag_cronologia_compativel")
    )
    .select(
        "id_fatura",
        "mes_referencia",
        "data_emissao",
        "_diag_formato_emissao",
        "_diag_emissao_cenario",
        "data_vencimento",
        "data_pagamento",
    )
    .limit(20)
)

_diag_mdy_compativel,_diag_dmy_compativel,quantidade
true,false,11905
true,true,8511


id_fatura,mes_referencia,data_emissao,_diag_formato_emissao,_diag_emissao_cenario,data_vencimento,data_pagamento


Das 20.416 datas ambíguas:
- 11.905: somente MDY respeita a cronologia testada.
- 8.511: ambas as interpretações respeitam a cronologia.

## 6. Diagnóstico dos valores monetários

Objetivos:
- identificar as representações monetárias;
- testar conversão para DECIMAL(18,2);
- medir valores não interpretados, zeros e negativos;
- verificar energia + tributos = total por fatura.

As colunas originais permanecem intactas.

As gramáticas testadas aceitam:
- decimal com ponto e duas casas;
- R$ com vírgula decimal, com ou sem agrupamento de milhares.

O sinal negativo é reconhecido para diagnóstico, sem converter o valor
para positivo. Zeros e negativos serão contados para avaliação.

Não removemos caracteres indiscriminadamente nem substituímos falhas por zero.

In [0]:
COLUNAS_MONETARIAS = [
    "valor_energia",
    "tributos",
    "valor_total",
]

PADRAO_DECIMAL = r"^-?[0-9]+\.[0-9]{2}$"
PADRAO_BRL_SIMPLES = r"^R\$ -?[0-9]+,[0-9]{2}$"
PADRAO_BRL_MILHARES = r"^R\$ -?[0-9]{1,3}(?:\.[0-9]{3})+,[0-9]{2}$"

df_monetario_diagnostico = df_faturas_analise

for coluna in COLUNAS_MONETARIAS:
    texto = F.col(coluna)

    formato = (
        F.when(texto.isNull(), "NULO")
        .when(F.trim(texto) == "", "VAZIO")
        .when(texto.rlike(PADRAO_DECIMAL), "DECIMAL_COM_PONTO")
        .when(texto.rlike(PADRAO_BRL_SIMPLES), "BRL_SEM_MILHARES")
        .when(texto.rlike(PADRAO_BRL_MILHARES), "BRL_COM_MILHARES")
        .otherwise("OUTRO_FORMATO")
    )

    eh_brl = (
        texto.rlike(PADRAO_BRL_SIMPLES)
        | texto.rlike(PADRAO_BRL_MILHARES)
    )

    texto_normalizado = (
        F.when(texto.rlike(PADRAO_DECIMAL), texto)
        .when(
            eh_brl,
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(texto, r"^R\$ ", ""),
                    r"\.",
                    "",
                ),
                ",",
                ".",
            ),
        )
    )

    df_monetario_diagnostico = (
        df_monetario_diagnostico
        .withColumn(f"_diag_formato_{coluna}", formato)
        .withColumn("_diag_texto_monetario", texto_normalizado)
        .withColumn(
            f"_diag_{coluna}",
            F.expr(
                "try_cast(_diag_texto_monetario AS DECIMAL(18,2))"
            ),
        )
        .drop("_diag_texto_monetario")
    )

Decisão técnica: só remover símbolos e separadores quando o texto corresponder a uma gramática conhecida. Um formato desconhecido permanece sem interpretação para inspeção.

In [0]:
for coluna in COLUNAS_MONETARIAS:
    valor = F.col(f"_diag_{coluna}")

    print(f"Diagnóstico de: {coluna}")

    display(
        df_monetario_diagnostico
        .groupBy(f"_diag_formato_{coluna}")
        .agg(
            F.count("*").alias("quantidade"),

            F.count(
                F.when(valor.isNull(), 1)
            ).alias("nao_interpretados"),

            F.count(
                F.when(valor == 0, 1)
            ).alias("zeros"),

            F.count(
                F.when(valor < 0, 1)
            ).alias("negativos"),

            F.min(valor).alias("menor_valor"),
            F.max(valor).alias("maior_valor"),
            F.sum(valor).alias("soma_interpretada"),
        )
        .orderBy(f"_diag_formato_{coluna}")
    )

Diagnóstico de: valor_energia


_diag_formato_valor_energia,quantidade,nao_interpretados,zeros,negativos,menor_valor,maior_valor,soma_interpretada
DECIMAL_COM_PONTO,2184621,0,0,0,4.78,35810.60,507506363.30


Diagnóstico de: tributos


_diag_formato_tributos,quantidade,nao_interpretados,zeros,negativos,menor_valor,maior_valor,soma_interpretada
DECIMAL_COM_PONTO,2184621,0,0,0,1.59,11936.87,169168784.28


Diagnóstico de: valor_total


_diag_formato_valor_total,quantidade,nao_interpretados,zeros,negativos,menor_valor,maior_valor,soma_interpretada
BRL_COM_MILHARES,8051,0,0,0,1000.11,41141.65,29366655.47
BRL_SEM_MILHARES,188571,0,0,0,7.72,999.96,31728807.46
DECIMAL_COM_PONTO,1987999,0,0,0,6.37,47747.47,615579684.65


As somas consideram somente os valores interpretados. Por isso, só poderemos usá-las como totais completos se nao_interpretados for zero em todos os grupos.
Valores altos, isoladamente, não justificam exclusão ou aplicação de um teto.

In [0]:
energia = F.col("_diag_valor_energia")
tributos = F.col("_diag_tributos")
total = F.col("_diag_valor_total")

campos_interpretados = (
    energia.isNotNull()
    & tributos.isNotNull()
    & total.isNotNull()
)

df_monetario_diagnostico = (
    df_monetario_diagnostico
    .withColumn(
        "_diag_diferenca_total",
        total - (energia + tributos),
    )
)

diferenca = F.col("_diag_diferenca_total")

display(
    df_monetario_diagnostico.agg(
        F.count("*").alias("registros"),

        F.count(
            F.when(~campos_interpretados, 1)
        ).alias("registros_com_falha_de_interpretacao"),

        F.count(
            F.when(
                campos_interpretados & (diferenca == 0),
                1,
            )
        ).alias("faturas_conciliadas"),

        F.count(
            F.when(
                campos_interpretados & (diferenca != 0),
                1,
            )
        ).alias("faturas_com_divergencia"),
    )
)

display(
    df_monetario_diagnostico
    .filter(
        ~campos_interpretados
        | (diferenca != 0)
    )
    .select(
        "id_fatura",
        "valor_energia",
        "tributos",
        "valor_total",
        "_diag_valor_energia",
        "_diag_tributos",
        "_diag_valor_total",
        "_diag_diferenca_total",
    )
    .limit(20)
)

registros,registros_com_falha_de_interpretacao,faturas_conciliadas,faturas_com_divergencia
2184621,0,2184621,0


id_fatura,valor_energia,tributos,valor_total,_diag_valor_energia,_diag_tributos,_diag_valor_total,_diag_diferenca_total


Tratamento fundamentado: converter os três campos para DECIMAL(18,2) pelas gramáticas verificadas e exigir conciliação exata por fatura. Não há necessidade de arredondar diferenças nem substituir valores.
O maior total é R$ 47.747,47. Ser alto não comprova erro; vamos preservá-lo. As verificações confirmam conversão e coerência aritmética, sem comprovar a legitimidade de cada cobrança.

### Transição para a análise de referências

Após a tipagem e a conciliação monetária, as próximas seções investigam a presença das chaves no cadastro e nas leituras, preservando a quantidade de faturas e os valores financeiros.


## 7. Identificadores e referência cadastral

Objetivos:
- fixar a versão Delta do cadastro usada na comparação;
- verificar os padrões dos identificadores;
- observar nulos, espaços e repetição de chaves no cadastro;
- medir faturas com e sem correspondência cadastral;
- verificar a relação entre ausência cadastral e múltiplas faturas por UC/mês.

O cadastro será usado apenas como referência de presença de identificadores.
Encontrar uma chave não comprova que todos os atributos cadastrais estejam tratados.

A projeção de chaves distintas evita multiplicação das faturas no join.
Isso não resolve duplicidades ou conflitos dos atributos do cadastro.

Nenhuma fatura será excluída, nenhum cadastro será criado e nenhuma tabela
será gravada nesta seção.

As duas fontes terão versões fixas, mas isso, isoladamente, não garante que pertençam à mesma extração. Vamos registrar também os metadados disponíveis.

In [0]:
bronze_cadastro = (
    f"`{CATALOGO}`.`{SCHEMA_BRONZE}`.`tb_unidades_consumidoras`"
)

historico_cadastro = spark.sql(
    f"DESCRIBE HISTORY {bronze_cadastro} LIMIT 1"
).select("version", "timestamp", "operation").first()

VERSAO_BRONZE_CADASTRO = historico_cadastro["version"]

df_cadastro_bronze = (
    spark.read
    .option("versionAsOf", VERSAO_BRONZE_CADASTRO)
    .table(bronze_cadastro)
)

print(f"Cadastro: {bronze_cadastro}")
print(f"Versão Delta: {VERSAO_BRONZE_CADASTRO}")
print(f"Timestamp do commit Delta: {historico_cadastro['timestamp']}")
print(f"Operação: {historico_cadastro['operation']}")

df_cadastro_bronze.printSchema()

display(
    df_cadastro_bronze
    .groupBy("source_file", "ingestion_datetime")
    .agg(F.count("*").alias("quantidade"))
)

Cadastro: `v-energy`.`bronze`.`tb_unidades_consumidoras`
Versão Delta: 0
Timestamp do commit Delta: 2026-10-22 18:38:55
Operação: CREATE OR REPLACE TABLE AS SELECT
root
 |-- id_uc: string (nullable = true)
 |-- cpf_cnpj: string (nullable = true)
 |-- nome_titular: string (nullable = true)
 |-- logradouro: string (nullable = true)
 |-- numero: string (nullable = true)
 |-- bairro: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- cep: string (nullable = true)
 |-- classe: string (nullable = true)
 |-- subgrupo_tarifario: string (nullable = true)
 |-- alimentador: string (nullable = true)
 |-- data_ligacao: string (nullable = true)
 |-- status: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



source_file,ingestion_datetime,quantidade
unidades_consumidoras.csv,2026-10-22T18:38:51.184Z,46461


Vamos testar os padrões FT ou FTX seguidos de números para faturas e UC seguido de números para unidades. São verificações de formato, sem excluir registros.

In [0]:
if df_cadastro_bronze.schema["id_uc"].dataType.simpleString() != "string":
    raise ValueError("Esperávamos id_uc como STRING na Bronze do cadastro.")

display(
    df_monetario_diagnostico.agg(
        F.count(
            F.when(
                F.col("id_fatura").isNull()
                | ~F.col("id_fatura").rlike(r"^FTX?[0-9]+$"),
                1,
            )
        ).alias("ids_fatura_fora_do_padrao"),

        F.count(
            F.when(
                F.col("id_uc").isNull()
                | ~F.col("id_uc").rlike(r"^UC[0-9]+$"),
                1,
            )
        ).alias("ids_uc_faturas_fora_do_padrao"),
    )
)

uc_cadastro = F.col("id_uc")

display(
    df_cadastro_bronze.agg(
        F.count("*").alias("linhas_cadastro"),

        F.count(
            F.when(uc_cadastro.isNull(), 1)
        ).alias("ids_nulos"),

        F.count(
            F.when(F.trim(uc_cadastro) == "", 1)
        ).alias("ids_vazios"),

        F.count(
            F.when(uc_cadastro != F.trim(uc_cadastro), 1)
        ).alias("ids_com_espacos_nas_extremidades"),

        F.count(
            F.when(uc_cadastro == "id_uc", 1)
        ).alias("valores_iguais_ao_nome_da_coluna"),

        F.count(
            F.when(
                uc_cadastro.isNotNull()
                & ~uc_cadastro.rlike(r"^UC[0-9]+$"),
                1,
            )
        ).alias("ids_fora_do_padrao"),
    )
)

# Exemplos de identificadores repetidos no cadastro bruto.
display(
    df_cadastro_bronze
    .groupBy("id_uc")
    .agg(F.count("*").alias("quantidade"))
    .filter(F.col("quantidade") > 1)
    .orderBy(F.desc("quantidade"))
    .limit(20)
)

# O que observar: espaços ou identificadores inesperados podem afetar a correspondência literal. Se aparecerem, avaliaremos seu efeito antes de concluir que todas as ausências são referências realmente faltantes.

ids_fatura_fora_do_padrao,ids_uc_faturas_fora_do_padrao
0,0


linhas_cadastro,ids_nulos,ids_vazios,ids_com_espacos_nas_extremidades,valores_iguais_ao_nome_da_coluna,ids_fora_do_padrao
46461,0,0,0,1,201


id_uc,quantidade
UC128167,3
UC120374,3
UC117503,3
UC111863,3
UC100381,2
UC100470,2
UC100494,2
UC100336,2
UC100583,2
UC100247,2


In [0]:
df_chaves_cadastro = (
    df_cadastro_bronze
    .select("id_uc")
    .filter(
        F.col("id_uc").isNotNull()
        & (F.trim("id_uc") != "")
        & (F.col("id_uc") != "id_uc")
    )
    .distinct()
    .withColumn("_diag_presente_cadastro", F.lit(True))
)

df_referencia_diagnostico = (
    df_monetario_diagnostico
    .join(df_chaves_cadastro, on="id_uc", how="left")
    .withColumn(
        "_diag_uc_sem_cadastro",
        F.col("_diag_presente_cadastro").isNull(),
    )
)

linhas_antes = df_monetario_diagnostico.count()
linhas_depois = df_referencia_diagnostico.count()

print(f"Faturas antes do join: {linhas_antes:,}")
print(f"Faturas depois do join: {linhas_depois:,}")

if linhas_antes != linhas_depois:
    raise ValueError("O cruzamento alterou a quantidade de faturas.")

display(
    df_referencia_diagnostico
    .groupBy("_diag_uc_sem_cadastro")
    .agg(
        F.count("*").alias("faturas"),
        F.countDistinct("id_uc").alias("ucs_distintas"),
        F.sum("_diag_valor_energia").alias("soma_energia"),
        F.sum("_diag_tributos").alias("soma_tributos"),
        F.sum("_diag_valor_total").alias("soma_total"),
    )
)

Faturas antes do join: 2,184,621
Faturas depois do join: 2,184,621


_diag_uc_sem_cadastro,faturas,ucs_distintas,soma_energia,soma_tributos,soma_total
true,32285,31705,7644280.57,2548093.25,10192373.82
false,2152336,45000,499862082.73,166620691.03,666482773.76


Precisamos descobrir se as 42 faturas dos pares UC/mês repetidos também estão no grupo sem cadastro. Isso evita contar a mesma fatura duas vezes ao medir problemas.

In [0]:
df_pares_multiplos = (
    df_monetario_diagnostico
    .groupBy("id_uc", "mes_referencia")
    .agg(F.count("*").alias("quantidade"))
    .filter(F.col("quantidade") > 1)
    .select("id_uc", "mes_referencia")
)

df_multiplas_com_referencia = (
    df_referencia_diagnostico
    .join(
        df_pares_multiplos,
        on=["id_uc", "mes_referencia"],
        how="inner",
    )
)

display(
    df_multiplas_com_referencia
    .groupBy("_diag_uc_sem_cadastro")
    .agg(
        F.count("*").alias("faturas_em_pares_multiplos"),
        F.countDistinct(
            "id_uc", "mes_referencia"
        ).alias("pares_uc_competencia"),
        F.sum("_diag_valor_total").alias("soma_total"),
    )
)

display(
    df_referencia_diagnostico
    .filter(F.col("_diag_uc_sem_cadastro"))
    .select(
        "id_fatura",
        "id_uc",
        "mes_referencia",
        "valor_total",
        "status",
    )
    .limit(10)
)

_diag_uc_sem_cadastro,faturas_em_pares_multiplos,pares_uc_competencia,soma_total
true,42,21,20341.61


id_fatura,id_uc,mes_referencia,valor_total,status
FTX0000000,UC9969129,2026-01,117.09,EM ABERTO
FTX0000001,UC9156745,2023-06,72.07,PAGA
FTX0000002,UC9707518,2024-04,360.59,PAGA
FTX0000003,UC9614202,2025-01,119.25,PAGA
FTX0000004,UC9292077,2024-02,214.68,EM ABERTO
FTX0000005,UC9348842,2024-04,488.51,PAGA
FTX0000006,UC9809874,2026-06,168.29,PAGA
FTX0000007,UC9868106,2025-12,108.45,PAGA
FTX0000008,UC9227071,2022-09,221.17,PAGA
FTX0000009,UC9348221,2023-12,122.00,PAGA


Os resultados mostram:

| Correspondência cadastral | Faturas | UCs distintas | Valor total |
|---|---:|---:|---:|
| Encontrada | 2.152.336 | 45.000 | R$ 666.482.773,76 |
| Não encontrada | 32.285 | 31.705 | R$ 10.192.373,82 |

As 42 faturas dos 21 pares UC/mês repetidos estão todas entre as 32.285 sem cadastro. Seus R$ 20.341,61 já estão incluídos nos R$ 10.192.373,82; não devemos somá-los novamente.
Também confirmamos:
- Todos os identificadores das faturas seguem os padrões testados.
- O cadastro possui chaves repetidas; usar chaves distintas foi necessário para evitar multiplicação no cruzamento.
- Existem 201 valores cadastrais fora do padrão, incluindo uma ocorrência literal de id_uc. Isso pertence ao diagnóstico do cadastro e não justifica alterar identificadores das faturas.

A recomendação para as faturas sem cadastro é preservá-las em uma saída de exceções, com motivo e reconciliação financeira, sem liberá-las às análises que exigem referência cadastral válida. Isso define seu destino no pipeline; não comprova que as cobranças sejam inválidas.

Para as múltiplas faturas, não recomendo excluir nenhuma como duplicata: seus IDs são distintos e não existe informação de cancelamento ou reemissão que permita escolher uma delas.

Antes de construir a candidata Silver, vamos verificar a relação com medição.

## 8. Diagnóstico da relação entre faturamento e medição

Objetivos:
- fixar a versão Delta de leituras;
- identificar cabeçalhos internos completos;
- medir a multiplicidade de leituras por UC/competência;
- verificar a presença das UCs/competências faturadas em leituras;
- cruzar essa cobertura com a correspondência cadastral.

A referência será somente um conjunto de pares UC/competência.
Não serão incorporados valores de consumo nem atributos de medidores.

Mais de uma leitura por UC/mês não será interpretada automaticamente
como duplicação: pode exigir avaliação do medidor e do tipo de leitura.

Nenhuma tabela será gravada e nenhuma fatura será excluída.

A existência do par UC/mês na Bronze de leituras não comprova que o consumo esteja tratado. Da mesma forma, ausência desse par não será uma regra automática de exclusão. Estamos investigando a relação para fundamentar o contrato.

In [0]:
bronze_leituras = (
    f"`{CATALOGO}`.`{SCHEMA_BRONZE}`.`tb_leituras`"
)

historico_leituras = spark.sql(
    f"DESCRIBE HISTORY {bronze_leituras} LIMIT 1"
).select("version", "timestamp", "operation").first()

VERSAO_BRONZE_LEITURAS = historico_leituras["version"]

df_leituras_bronze = (
    spark.read
    .option("versionAsOf", VERSAO_BRONZE_LEITURAS)
    .table(bronze_leituras)
)

print(f"Origem: {bronze_leituras}")
print(f"Versão Delta: {VERSAO_BRONZE_LEITURAS}")
print(f"Timestamp do commit Delta: {historico_leituras['timestamp']}")
print(f"Operação: {historico_leituras['operation']}")

df_leituras_bronze.printSchema()

display(
    df_leituras_bronze
    .groupBy("source_file", "ingestion_datetime")
    .agg(F.count("*").alias("quantidade"))
    .orderBy("source_file")
)

Origem: `v-energy`.`bronze`.`tb_leituras`
Versão Delta: 0
Timestamp do commit Delta: 2026-10-22 18:40:16
Operação: CREATE OR REPLACE TABLE AS SELECT
root
 |-- id_leitura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- id_medidor: string (nullable = true)
 |-- data_leitura: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- leitura_anterior: string (nullable = true)
 |-- leitura_atual: string (nullable = true)
 |-- consumo_kwh: string (nullable = true)
 |-- tipo_leitura: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)



source_file,ingestion_datetime,quantidade
leituras_2022.csv,2026-10-22T18:40:12.034Z,271744
leituras_2023.csv,2026-10-22T18:40:12.034Z,544546
leituras_2024.csv,2026-10-22T18:40:12.034Z,545652
leituras_2025.csv,2026-10-22T18:40:12.034Z,546908
leituras_2026.csv,2026-10-22T18:40:12.034Z,273618


In [0]:
COLUNAS_NEGOCIO_LEITURAS = [
    "id_leitura",
    "id_uc",
    "id_medidor",
    "data_leitura",
    "mes_referencia",
    "leitura_anterior",
    "leitura_atual",
    "consumo_kwh",
    "tipo_leitura",
]

ausentes = (
    set(COLUNAS_NEGOCIO_LEITURAS)
    - set(df_leituras_bronze.columns)
)

if ausentes:
    raise ValueError(f"Colunas ausentes em leituras: {sorted(ausentes)}")

eh_cabecalho_leitura = F.lit(True)

for coluna in COLUNAS_NEGOCIO_LEITURAS:
    eh_cabecalho_leitura = (
        eh_cabecalho_leitura
        & F.col(coluna).eqNullSafe(F.lit(coluna))
    )

df_leituras_rotuladas = df_leituras_bronze.withColumn(
    "_diag_cabecalho",
    eh_cabecalho_leitura,
)

display(
    df_leituras_rotuladas
    .groupBy("_diag_cabecalho")
    .agg(F.count("*").alias("quantidade"))
)

df_leituras_negocio = (
    df_leituras_rotuladas
    .filter(~F.col("_diag_cabecalho"))
    .drop("_diag_cabecalho")
)

display(
    df_leituras_negocio.agg(
        F.count("*").alias("registros"),

        F.count(
            F.when(
                F.col("id_uc").isNull()
                | ~F.col("id_uc").rlike(r"^UC[0-9]+$"),
                1,
            )
        ).alias("ids_uc_ausentes_ou_fora_do_padrao"),

        F.count(
            F.when(
                F.col("mes_referencia").isNull()
                | ~F.col("mes_referencia").rlike(
                    r"^[1-9][0-9]{3}-(0[1-9]|1[0-2])$"
                ),
                1,
            )
        ).alias("competencias_ausentes_ou_invalidas"),
    )
)

_diag_cabecalho,quantidade
false,2182468


registros,ids_uc_ausentes_ou_fora_do_padrao,competencias_ausentes_ou_invalidas
2182468,0,0


In [0]:
df_pares_leituras = (
    df_leituras_negocio
    .filter(
        F.col("id_uc").isNotNull()
        & F.col("mes_referencia").isNotNull()
    )
    .groupBy("id_uc", "mes_referencia")
    .agg(F.count("*").alias("quantidade_leituras"))
)

display(
    df_pares_leituras.agg(
        F.count("*").alias("pares_uc_competencia"),

        F.count(
            F.when(F.col("quantidade_leituras") > 1, 1)
        ).alias("pares_com_multiplas_leituras"),

        F.max("quantidade_leituras").alias(
            "maior_quantidade_por_par"
        ),
    )
)

# O agrupamento já garante uma linha por par.
df_chaves_leituras = (
    df_pares_leituras
    .select("id_uc", "mes_referencia")
    .withColumn("_diag_leitura_presente", F.lit(True))
)

pares_uc_competencia,pares_com_multiplas_leituras,maior_quantidade_por_par
2152336,30061,3


Decisão técnica: cruzar com uma linha por chave de referência. Não vamos juntar diretamente todas as leituras às faturas, pois isso poderia repetir valores financeiros.

In [0]:
df_referencias_completas = (
    df_referencia_diagnostico
    .join(
        df_chaves_leituras,
        on=["id_uc", "mes_referencia"],
        how="left",
    )
    .withColumn(
        "_diag_sem_leitura_uc_mes",
        F.col("_diag_leitura_presente").isNull(),
    )
)

antes = df_referencia_diagnostico.count()
depois = df_referencias_completas.count()

print(f"Faturas antes do cruzamento: {antes:,}")
print(f"Faturas depois do cruzamento: {depois:,}")

if antes != depois:
    raise ValueError("O cruzamento com leituras alterou a quantidade de faturas.")

display(
    df_referencias_completas
    .groupBy(
        "_diag_uc_sem_cadastro",
        "_diag_sem_leitura_uc_mes",
    )
    .agg(
        F.count("*").alias("faturas"),
        F.countDistinct("id_uc").alias("ucs_distintas"),
        F.sum("_diag_valor_total").alias("soma_total"),
    )
    .orderBy(
        "_diag_uc_sem_cadastro",
        "_diag_sem_leitura_uc_mes",
    )
)

display(
    df_referencias_completas
    .filter(
        ~F.col("_diag_uc_sem_cadastro")
        & F.col("_diag_sem_leitura_uc_mes")
    )
    .select(
        "id_fatura",
        "id_uc",
        "mes_referencia",
        "valor_total",
        "status",
    )
    .limit(20)
)

Faturas antes do cruzamento: 2,184,621
Faturas depois do cruzamento: 2,184,621


_diag_uc_sem_cadastro,_diag_sem_leitura_uc_mes,faturas,ucs_distintas,soma_total
false,false,2152336,45000,666482773.76
true,true,32285,31705,10192373.82


id_fatura,id_uc,mes_referencia,valor_total,status


O cruzamento preservou as 2.184.621 faturas e mostrou:

| Situação | Faturas | Valor total |
|---|---:|---:|
| Com cadastro e leitura na competência | 2.152.336 | R$ 666.482.773,76 |
| Sem cadastro e sem leitura na competência | 32.285 | R$ 10.192.373,82 |
| Com cadastro, mas sem leitura na competência | 0 | — |

Em leituras, existem 30.061 pares UC/mês com mais de um registro, chegando a três leituras por par. Isso reforça a necessidade de cruzar com chaves distintas, sem multiplicar as faturas. Não precisamos deduplicar as leituras para verificar a existência do par.

Agora temos evidências suficientes para construir uma candidata Silver em memória. As referências verificadas ainda são da Bronze; encontrar as chaves não homologa os atributos cadastrais nem os valores de consumo.

## 9. Candidata Silver de faturas — somente em memória

Esta etapa aplica as regras propostas e fundamentadas na exploração.

Preservação:
- as onze colunas originais ficam em dados_originais;
- source_file e ingestion_datetime permanecem;
- versões Delta das três fontes são registradas;
- o método de interpretação da emissão é explícito.

A ambiguidade original permanece sinalizada mesmo quando a convenção MDY
produz uma data escolhida.

As faturas são preservadas integralmente. A separação cadastral cria
DataFrames para revisão, sem gravar tabelas.

Cadastro encontrado não significa liberação automática à Gold.
As validações da candidata serão feitas na próxima seção.

As escolhas aplicadas nesta candidata ficam explícitas:

| Tema | Tratamento |
|---|---|
| Cabeçalhos internos | Usar somente os registros de negócio já separados |
| Valores monetários | Converter para `DECIMAL(18,2)` |
| Emissão com hífen e ano no fim | Adotar MDY como convenção do projeto |
| Emissão com barras ou pontos | Manter a hipótese DMY, documentando o método |
| Pagamento nulo ou sentinela | Representar como `NULL`, preservando o motivo |
| Emissão fora da competência e pagamento atrasado | Preservar e sinalizar |
| Múltiplas faturas por UC/mês | Preservar todas e sinalizar |
| Ausência cadastral | Separar uma prévia das pendências, sem descartar registros |
| Status e bandeira | Manter os valores observados; não há variações que exijam correção |

In [0]:
# Não vamos escolher datas pela proximidade com a competência. A emissão seguirá a máscara definida para cada representação.
# A emissão segue a máscara definida para cada representação.
# Pagamento e vencimento reutilizam a interpretação ISO do diagnóstico.

def converter_data(texto, mascara):
    return F.to_date(
        F.try_to_timestamp(texto, F.lit(mascara))
    )


texto_emissao = F.col("data_emissao")

eh_iso = texto_emissao.rlike(r"^\d{4}-\d{2}-\d{2}$")
eh_barras = texto_emissao.rlike(r"^\d{2}/\d{2}/\d{4}$")
eh_pontos = texto_emissao.rlike(r"^\d{2}\.\d{2}\.\d{4}$")
eh_hifen_final = texto_emissao.rlike(r"^\d{2}-\d{2}-\d{4}$")

emissao_mdy = converter_data(texto_emissao, "MM-dd-yyyy")
emissao_dmy = converter_data(texto_emissao, "dd-MM-yyyy")

emissao_escolhida = (
    F.when(
        eh_iso,
        converter_data(texto_emissao, "yyyy-MM-dd"),
    )
    .when(
        eh_barras,
        converter_data(texto_emissao, "dd/MM/yyyy"),
    )
    .when(
        eh_pontos,
        converter_data(texto_emissao, "dd.MM.yyyy"),
    )
    .when(
        eh_hifen_final,
        emissao_mdy,
    )
)

metodo_emissao = (
    F.when(eh_iso, "ISO_ANO_MES_DIA")
    .when(eh_barras, "CONVENCAO_DMY_BARRAS")
    .when(eh_pontos, "CONVENCAO_DMY_PONTOS")
    .when(eh_hifen_final, "CONVENCAO_MDY_HIFEN")
    .otherwise("NAO_RECONHECIDO")
)

ambiguidade_original = F.coalesce(
    eh_hifen_final & (emissao_mdy != emissao_dmy),
    F.lit(False),
)

motivo_ausencia_pagamento = (
    F.when(
        F.col("data_pagamento").isNull(),
        "CAMPO_NULO",
    )
    .when(
        F.col("data_pagamento") == "9999-12-31",
        "SENTINELA_9999_12_31",
    )
)

pagamento_tipado = F.when(
    F.col("data_pagamento") != "9999-12-31",
    interpretar_data_iso("data_pagamento"),
)

df_faturas_candidata = df_referencias_completas.select(
    "id_fatura",
    "id_uc",
    "mes_referencia",

    emissao_escolhida.alias("data_emissao"),

    F.col("_diag_valor_energia").alias("valor_energia"),
    F.col("_diag_tributos").alias("tributos"),
    "bandeira_tarifaria",
    F.col("_diag_valor_total").alias("valor_total"),

    interpretar_data_iso("data_vencimento").alias(
        "data_vencimento"
    ),

    pagamento_tipado.alias("data_pagamento"),
    "status",

    metodo_emissao.alias("metodo_interpretacao_emissao"),
    ambiguidade_original.alias(
        "flag_emissao_original_ambigua"
    ),
    motivo_ausencia_pagamento.alias(
        "motivo_ausencia_pagamento"
    ),

    F.col("_diag_uc_sem_cadastro").alias(
        "flag_uc_sem_cadastro"
    ),
    F.col("_diag_sem_leitura_uc_mes").alias(
        "flag_sem_leitura_uc_mes"
    ),

    F.struct(*[
        F.col(coluna)
        for coluna in COLUNAS_NEGOCIO
    ]).alias("dados_originais"),

    "source_file",
    "ingestion_datetime",
)

Decisão importante: a data escolhida fica disponível como DATE, mas flag_emissao_original_ambigua continua verdadeira nos 20.416 casos. Isso distingue uma convenção adotada de uma informação confirmada pela origem

In [0]:
from pyspark.sql import Window

janela_uc_mes = Window.partitionBy(
    "id_uc", "mes_referencia"
)

df_faturas_candidata = (
    df_faturas_candidata
    .withColumn(
        "flag_multiplas_faturas_uc_mes",
        F.count("*").over(janela_uc_mes) > 1,
    )
    .withColumn(
        "flag_emissao_fora_competencia",
        F.date_format("data_emissao", "yyyy-MM")
        != F.col("mes_referencia"),
    )
    .withColumn(
        "flag_pagamento_atrasado",
        F.coalesce(
            F.col("data_pagamento") > F.col("data_vencimento"),
            F.lit(False),
        ),
    )
    .withColumn(
        "versoes_bronze",
        F.struct(
            F.lit(VERSAO_BRONZE_FATURAS).alias("faturas"),
            F.lit(VERSAO_BRONZE_CADASTRO).alias("cadastro"),
            F.lit(VERSAO_BRONZE_LEITURAS).alias("leituras"),
        ),
    )
    .withColumn(
        "versao_regras",
        F.lit("FATURAS_CANDIDATA_V1"),
    )
    .withColumn(
        "situacao_referencia_cadastral",
        F.when(
            F.col("flag_uc_sem_cadastro"),
            "PENDENTE_CADASTRO",
        ).otherwise("CADASTRO_ENCONTRADO"),
    )
)

df_faturas_com_cadastro = df_faturas_candidata.filter(
    ~F.col("flag_uc_sem_cadastro")
)

df_faturas_pendentes_cadastro = df_faturas_candidata.filter(
    F.col("flag_uc_sem_cadastro")
)

O que essa separação faz: mantém todas as faturas disponíveis e cria uma prévia do destino das pendências cadastrais. As 42 faturas dos pares repetidos continuam preservadas nesse grupo, sem uma segunda exclusão.

In [0]:
df_faturas_candidata.printSchema()

display(
    df_faturas_candidata
    .groupBy("situacao_referencia_cadastral")
    .agg(
        F.count("*").alias("faturas"),
        F.sum("valor_energia").alias("soma_energia"),
        F.sum("tributos").alias("soma_tributos"),
        F.sum("valor_total").alias("soma_total"),
    )
)

indicadores = [
    "flag_emissao_original_ambigua",
    "flag_emissao_fora_competencia",
    "flag_pagamento_atrasado",
    "flag_uc_sem_cadastro",
    "flag_sem_leitura_uc_mes",
    "flag_multiplas_faturas_uc_mes",
]

display(
    df_faturas_candidata.agg(*[
        F.count(
            F.when(F.col(indicador), 1)
        ).alias(indicador)
        for indicador in indicadores
    ])
)

display(
    df_faturas_candidata.select(
        "id_fatura",
        "dados_originais.data_emissao",
        F.col("data_emissao").alias("data_emissao_tipada"),
        "metodo_interpretacao_emissao",
        "flag_emissao_original_ambigua",
        "valor_total",
        "situacao_referencia_cadastral",
    ).limit(10)
)

root
 |-- id_fatura: string (nullable = true)
 |-- id_uc: string (nullable = true)
 |-- mes_referencia: string (nullable = true)
 |-- data_emissao: date (nullable = true)
 |-- valor_energia: decimal(18,2) (nullable = true)
 |-- tributos: decimal(18,2) (nullable = true)
 |-- bandeira_tarifaria: string (nullable = true)
 |-- valor_total: decimal(18,2) (nullable = true)
 |-- data_vencimento: date (nullable = true)
 |-- data_pagamento: date (nullable = true)
 |-- status: string (nullable = true)
 |-- metodo_interpretacao_emissao: string (nullable = false)
 |-- flag_emissao_original_ambigua: boolean (nullable = false)
 |-- motivo_ausencia_pagamento: string (nullable = true)
 |-- flag_uc_sem_cadastro: boolean (nullable = false)
 |-- flag_sem_leitura_uc_mes: boolean (nullable = false)
 |-- dados_originais: struct (nullable = false)
 |    |-- id_fatura: string (nullable = true)
 |    |-- id_uc: string (nullable = true)
 |    |-- mes_referencia: string (nullable = true)
 |    |-- data_emissao: 

situacao_referencia_cadastral,faturas,soma_energia,soma_tributos,soma_total
CADASTRO_ENCONTRADO,2152336,499862082.73,166620691.03,666482773.76
PENDENTE_CADASTRO,32285,7644280.57,2548093.25,10192373.82


flag_emissao_original_ambigua,flag_emissao_fora_competencia,flag_pagamento_atrasado,flag_uc_sem_cadastro,flag_sem_leitura_uc_mes,flag_multiplas_faturas_uc_mes
20416,11306,1675636,32285,32285,42


id_fatura,data_emissao,data_emissao_tipada,metodo_interpretacao_emissao,flag_emissao_original_ambigua,valor_total,situacao_referencia_cadastral
FT000000000,2022-07-27,2022-07-27,ISO_ANO_MES_DIA,false,102.57,CADASTRO_ENCONTRADO
FT000000001,2022-08-11,2022-08-11,ISO_ANO_MES_DIA,false,115.60,CADASTRO_ENCONTRADO
FT000000002,2022-09-12,2022-09-12,ISO_ANO_MES_DIA,false,129.91,CADASTRO_ENCONTRADO
FT000000003,2022-10-28,2022-10-28,ISO_ANO_MES_DIA,false,167.93,CADASTRO_ENCONTRADO
FT000000004,2022-11-24,2022-11-24,ISO_ANO_MES_DIA,false,234.19,CADASTRO_ENCONTRADO
FT000000005,2022-12-05,2022-12-05,ISO_ANO_MES_DIA,false,309.68,CADASTRO_ENCONTRADO
FT000000006,2023-01-19,2023-01-19,ISO_ANO_MES_DIA,false,253.76,CADASTRO_ENCONTRADO
FT000000007,2023-02-06,2023-02-06,ISO_ANO_MES_DIA,false,298.07,CADASTRO_ENCONTRADO
FT000000008,2023-03-10,2023-03-10,ISO_ANO_MES_DIA,false,256.56,CADASTRO_ENCONTRADO
FT000000009,2023-04-30,2023-04-30,ISO_ANO_MES_DIA,false,255.61,CADASTRO_ENCONTRADO


A candidata apresentou os tipos esperados: datas como DATE, dinheiro como DECIMAL(18,2) e originais preservados em STRUCT.

As contagens também conferem: 20.416 ambiguidades originais, 11.306 emissões fora da competência, 1.675.636 pagamentos atrasados, 32.285 pendências cadastrais e 42 faturas em pares repetidos.

O nullable = true do schema indica que uma coluna pode conter nulos; não demonstra que existam nulos nos registros. Vamos verificar isso agora, junto com preservação e reconciliação.

## 10. Validação da candidata Silver

Verificações:
1. Tipos, campos obrigatórios, domínios e cronologia.
2. Preservação dos originais e comparação dos valores tipados
   com os diagnósticos anteriores.
3. Separação cadastral sem perda, acréscimo ou sobreposição de IDs.
4. Conservação dos valores por competência e status.

Uma falha interrompe a respectiva célula e exige investigação.
Não corrigimos os dados automaticamente para fazer a validação passar.

A aprovação desta seção valida a candidata em memória segundo o contrato
adotado. Não representa gravação nem execução do Workflow completo.

### Bloqueio de pagamentos não interpretados

Uma data de pagamento inválida pode virar NULL durante a conversão.
Por isso, conferir apenas o status e a data tipada não é suficiente.

A ausência reconhecida possui um motivo explícito:

- Pagamento originalmente nulo: `CAMPO_NULO`.
- Sentinela `9999-12-31`: `SENTINELA_9999_12_31`.

A regra `pagamento_nao_interpretado` identifica quando a data tipada
e o motivo de ausência estão ambos nulos. Nesse caso, a validação
interrompe a candidata.

Isso impede que calendário inválido, formato desconhecido ou texto
vazio sejam aceitos como ausência legítima de pagamento.

As regras anteriores continuam valendo:

- Fatura `PAGA` precisa de pagamento válido.
- Fatura `EM ABERTO` não pode apresentar pagamento válido.

A correção não modifica os valores originais nem grava tabelas.
Os outputs históricos anteriores à alteração não comprovam a execução
dessa nova regra; a conferência ocorre ao executar novamente a Seção 10.

In [0]:
VALIDACOES_CANDIDATA_OK = False
etapas_validadas = set()

tipos_esperados = {
    "id_fatura": "string",
    "id_uc": "string",
    "mes_referencia": "string",
    "data_emissao": "date",
    "valor_energia": "decimal(18,2)",
    "tributos": "decimal(18,2)",
    "bandeira_tarifaria": "string",
    "valor_total": "decimal(18,2)",
    "data_vencimento": "date",
    "data_pagamento": "date",
    "status": "string",
}

for coluna, esperado in tipos_esperados.items():
    obtido = df_faturas_candidata.schema[coluna].dataType.simpleString()
    if obtido != esperado:
        raise ValueError(f"{coluna}: esperado {esperado}, obtido {obtido}")

obrigatorias = [
    c for c in COLUNAS_NEGOCIO
    if c != "data_pagamento"
] + [
    "source_file",
    "ingestion_datetime",
    "metodo_interpretacao_emissao",
    "versao_regras",
    "situacao_referencia_cadastral",
]

regras = {
    f"{c}_nulo": F.col(c).isNull()
    for c in obrigatorias + indicadores
}

regras.update({
    "id_fatura_invalido":
        ~F.col("id_fatura").rlike(r"^FTX?[0-9]+$"),

    "id_uc_invalido":
        ~F.col("id_uc").rlike(r"^UC[0-9]+$"),

    "competencia_invalida":
        ~F.col("mes_referencia").rlike(
            r"^[1-9][0-9]{3}-(0[1-9]|1[0-2])$"
        ),

    "status_invalido":
        ~F.col("status").isin("PAGA", "EM ABERTO"),

    "bandeira_invalida":
        ~F.col("bandeira_tarifaria").isin(
            "Verde", "Amarela", "Vermelha P1", "Vermelha P2"
        ),

    "emissao_apos_vencimento":
        F.col("data_emissao") > F.col("data_vencimento"),

    "pagamento_antes_emissao":
        F.col("data_pagamento") < F.col("data_emissao"),

    "paga_sem_pagamento":
        (F.col("status") == "PAGA")
        & F.col("data_pagamento").isNull(),

    "em_aberto_com_pagamento":
        (F.col("status") == "EM ABERTO")
        & F.col("data_pagamento").isNotNull(),

    # Ausência reconhecida tem motivo; falha de parsing não pode passar como ausência.
    "pagamento_nao_interpretado": (
        F.col("data_pagamento").isNull()
        & F.col("motivo_ausencia_pagamento").isNull()
    ),

    "total_divergente":
        F.col("valor_energia") + F.col("tributos")
        != F.col("valor_total"),
})

for coluna in COLUNAS_MONETARIAS:
    regras[f"{coluna}_nao_positivo"] = F.col(coluna) <= 0

resultado_regras = df_faturas_candidata.agg(*[
    F.count(F.when(condicao, 1)).alias(nome)
    for nome, condicao in regras.items()
]).first().asDict()

falhas = {
    nome: quantidade
    for nome, quantidade in resultado_regras.items()
    if quantidade != 0
}

chaves = df_faturas_candidata.agg(
    F.count("*").alias("registros"),
    F.countDistinct("id_fatura").alias("ids_distintos"),
).first()

if falhas:
    raise ValueError(f"Falhas de qualidade: {falhas}")

if not chaves["registros"] or chaves["registros"] != chaves["ids_distintos"]:
    raise ValueError("Candidata vazia ou com IDs de fatura repetidos.")

etapas_validadas.add("contrato")
print("OK: tipos, chaves e regras por registro.")

OK: tipos, chaves e regras por registro.


Decisão aplicada: valores positivos e os domínios observados passam a integrar o contrato desta candidata. Se uma nova entrega trouxer zero, negativos ou outra categoria, a carga precisará de avaliação, em vez de alterar silenciosamente o significado do dado.

In [0]:
# Usaremos exceptAll nas duas direções. Diferentemente de uma comparação apenas de contagens, isso verifica o conteúdo com multiplicidade.

def conferir_mesmos_registros(esquerda, direita, colunas, descricao):
    a = esquerda.select(*colunas)
    b = direita.select(*colunas)

    if a.exceptAll(b).limit(1).count():
        raise ValueError(f"{descricao}: há registros exclusivos à esquerda.")

    if b.exceptAll(a).limit(1).count():
        raise ValueError(f"{descricao}: há registros exclusivos à direita.")

    print(f"OK: {descricao}.")


originais_recuperados = df_faturas_candidata.select(
    "dados_originais.*",
    "source_file",
    "ingestion_datetime",
)

conferir_mesmos_registros(
    originais_recuperados,
    df_faturas_analise,
    COLUNAS_NEGOCIO + ["source_file", "ingestion_datetime"],
    "Preservação dos registros originais e metadados",
)

# Referência financeira produzida durante a exploração.
referencia_financeira = df_monetario_diagnostico.select(
    "id_fatura",
    "id_uc",
    "mes_referencia",
    "bandeira_tarifaria",
    "status",
    *[
        F.col(f"_diag_{c}").alias(c)
        for c in COLUNAS_MONETARIAS
    ],
)

# Referência de datas do cenário MDY já avaliado.
referencia_datas = (
    df_cenarios_emissao
    .filter(F.col("_diag_cenario") == "HIFEN_COMO_MDY")
    .select(
        "id_fatura",
        F.col("_diag_emissao_cenario").alias("data_emissao"),
        F.col("_diag_vencimento").alias("data_vencimento"),
        F.col("_diag_pagamento").alias("data_pagamento"),
    )
)

referencia_tipada = referencia_financeira.join(
    referencia_datas,
    on="id_fatura",
    how="inner",
)

conferir_mesmos_registros(
    df_faturas_candidata,
    referencia_tipada,
    COLUNAS_NEGOCIO,
    "Valores tipados em relação aos diagnósticos anteriores",
)

etapas_validadas.add("conteudo")

# # Essa comparação verifica a aplicação do contrato escolhido, incluindo as datas MDY. Ela não transforma a convenção em confirmação documental da origem.

OK: Preservação dos registros originais e metadados.
OK: Valores tipados em relação aos diagnósticos anteriores.


In [0]:
ids_com_cadastro = df_faturas_com_cadastro.select("id_fatura")
ids_pendentes = df_faturas_pendentes_cadastro.select("id_fatura")

if ids_com_cadastro.join(
    ids_pendentes,
    on="id_fatura",
    how="inner",
).limit(1).count():
    raise ValueError("Há IDs presentes nas duas saídas.")

df_particoes_reunidas = (
    df_faturas_com_cadastro
    .unionByName(df_faturas_pendentes_cadastro)
)

conferir_mesmos_registros(
    df_particoes_reunidas,
    df_faturas_candidata,
    df_faturas_candidata.columns,
    "Reunião das saídas cadastrais",
)

if df_faturas_com_cadastro.filter(
    F.col("flag_uc_sem_cadastro")
).limit(1).count():
    raise ValueError("A saída com cadastro contém pendências cadastrais.")

if df_faturas_pendentes_cadastro.filter(
    ~F.col("flag_uc_sem_cadastro")
).limit(1).count():
    raise ValueError("A saída pendente contém faturas com cadastro.")

etapas_validadas.add("particoes")
print("OK: saídas sem sobreposição, perdas ou acréscimos.")

# Aqui comparamos todas as colunas das saídas reunidas com a candidata, incluindo os originais, indicadores e metadados.

OK: Reunião das saídas cadastrais.
OK: saídas sem sobreposição, perdas ou acréscimos.


In [0]:
# Além dos testes por registro, vamos conferir quantidade e dinheiro em cada grupo. Isso facilita identificar impactos localizados, que uma soma global poderia esconder.

grupos_reconciliacao = ["mes_referencia", "status"]


def resumir_financeiro(df, usar_diagnostico=False):
    return df.groupBy(*grupos_reconciliacao).agg(
        F.count("*").alias("faturas"),
        *[
            F.sum(
                F.col(f"_diag_{c}" if usar_diagnostico else c)
            ).alias(f"soma_{c}")
            for c in COLUNAS_MONETARIAS
        ],
    )


resumo_origem = resumir_financeiro(
    df_monetario_diagnostico,
    usar_diagnostico=True,
)

resumo_candidata = resumir_financeiro(df_faturas_candidata)
resumo_particoes = resumir_financeiro(df_particoes_reunidas)

colunas_resumo = (
    grupos_reconciliacao
    + ["faturas"]
    + [f"soma_{c}" for c in COLUNAS_MONETARIAS]
)

conferir_mesmos_registros(
    resumo_candidata,
    resumo_origem,
    colunas_resumo,
    "Conservação por competência e status",
)

conferir_mesmos_registros(
    resumo_particoes,
    resumo_candidata,
    colunas_resumo,
    "Conservação financeira após a separação",
)

display(
    df_faturas_candidata
    .groupBy("situacao_referencia_cadastral")
    .agg(
        F.count("*").alias("faturas"),
        *[
            F.sum(c).alias(f"soma_{c}")
            for c in COLUNAS_MONETARIAS
        ],
    )
)

etapas_validadas.add("conservacao")

if etapas_validadas != {
    "contrato", "conteudo", "particoes", "conservacao"
}:
    raise ValueError("Nem todas as etapas de validação foram concluídas.")

VALIDACOES_CANDIDATA_OK = True
print("OK: candidata validada em memória. Nenhuma tabela foi gravada.")

OK: Conservação por competência e status.
OK: Conservação financeira após a separação.


situacao_referencia_cadastral,faturas,soma_valor_energia,soma_tributos,soma_valor_total
CADASTRO_ENCONTRADO,2152336,499862082.73,166620691.03,666482773.76
PENDENTE_CADASTRO,32285,7644280.57,2548093.25,10192373.82


OK: candidata validada em memória. Nenhuma tabela foi gravada.


| Saída em memória | Faturas | Valor total |
|---|---:|---:|
| Com cadastro | 2.152.336 | R$ 666.482.773,76 |
| Pendentes de cadastro | 32.285 | R$ 10.192.373,82 |
| Total preservado | 2.184.621 | R$ 676.675.147,58 |

A candidata preserva os registros originais, aplica a tipagem conforme os diagnósticos e mantém quantidade e valores após a separação.

Agora precisamos verificar os destinos existentes. A próxima seção continua somente com consultas. Não grava tabelas nem faz commits.

## 11. Inspeção dos destinos

Objetivos:
- verificar a existência do schema Silver;
- identificar a tabela atual de faturas;
- comparar seu schema com a candidata;
- verificar se os nomes propostos para revisão estão disponíveis.

Esta seção não cria schema, não grava tabelas e não altera a Silver existente.

A persistência em tabelas de revisão será uma etapa separada.

A ideia é persistir primeiro em duas tabelas de revisão, para conferir o resultado salvo antes de decidir a substituição da Silver atual:
- silver.tb_faturas_revisao_v1
- silver.tb_faturas_pendentes_cadastro_revisao_v1
A primeira receberia as 2.152.336 faturas com cadastro; a segunda, as 32.285 pendentes. Esses nomes ainda serão apenas inspecionados.

In [0]:
if not globals().get("VALIDACOES_CANDIDATA_OK", False):
    raise ValueError(
        "A validação completa precisa estar concluída nesta sessão. "
        "Confira o final da Seção 10."
    )

SCHEMA_SILVER = "silver"
schema_silver = f"`{CATALOGO}`.`{SCHEMA_SILVER}`"

tabela_faturas_atual = f"{schema_silver}.`tb_faturas`"

destinos_revisao = [
    (
        f"{schema_silver}.`tb_faturas_revisao_v1`",
        df_faturas_com_cadastro,
    ),
    (
        f"{schema_silver}.`tb_faturas_pendentes_cadastro_revisao_v1`",
        df_faturas_pendentes_cadastro,
    ),
]

print(f"Tabela atual a inspecionar: {tabela_faturas_atual}")

for destino, dataframe in destinos_revisao:
    print(f"Destino proposto: {destino}")

print("Nenhuma gravação será executada nesta seção.")

Tabela atual a inspecionar: `v-energy`.`silver`.`tb_faturas`
Destino proposto: `v-energy`.`silver`.`tb_faturas_revisao_v1`
Destino proposto: `v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`
Nenhuma gravação será executada nesta seção.


In [0]:
# Se algum nome de revisão já estiver ocupado, registraremos essa informação antes de preparar qualquer escrita.

schema_silver_existe = (
    spark.sql(
        f"SHOW SCHEMAS IN `{CATALOGO}` LIKE '{SCHEMA_SILVER}'"
    )
    .limit(1)
    .count() > 0
)

print(f"Schema Silver existe: {schema_silver_existe}")

existencia_destinos = {}

if schema_silver_existe:
    display(
        spark.sql(f"SHOW TABLES IN {schema_silver}")
        .select("tableName", "isTemporary")
        .orderBy("tableName")
    )

for tabela in [
    tabela_faturas_atual,
    *[destino for destino, _ in destinos_revisao],
]:
    existe = (
        spark.catalog.tableExists(tabela)
        if schema_silver_existe
        else False
    )

    existencia_destinos[tabela] = existe
    print(f"{tabela}: {'EXISTE' if existe else 'NÃO EXISTE'}")

Schema Silver existe: True


tableName,isTemporary
tb_faturas,false
tb_faturas_pendentes_cadastro,false
tb_faturas_pendentes_cadastro_revisao_v1,false
tb_faturas_revisao_v1,false


`v-energy`.`silver`.`tb_faturas`: EXISTE
`v-energy`.`silver`.`tb_faturas_revisao_v1`: EXISTE
`v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`: EXISTE


In [0]:
# A separação cadastral altera a cobertura da tabela principal. Além dessa mudança de conteúdo, vamos verificar quais colunas seriam acrescentadas, retiradas ou teriam tipos diferentes.

def comparar_schemas(df_atual, df_proposto):
    atual = {
        campo.name: campo.dataType.simpleString()
        for campo in df_atual.schema.fields
    }

    proposto = {
        campo.name: campo.dataType.simpleString()
        for campo in df_proposto.schema.fields
    }

    diferencas = [
        (
            coluna,
            atual.get(coluna, "AUSENTE"),
            proposto.get(coluna, "AUSENTE"),
        )
        for coluna in sorted(set(atual) | set(proposto))
        if atual.get(coluna) != proposto.get(coluna)
    ]

    if diferencas:
        display(
            spark.createDataFrame(
                diferencas,
                "coluna string, tipo_atual string, tipo_proposto string",
            )
        )
    else:
        print("Nenhuma diferença de nomes ou tipos de colunas.")


if existencia_destinos[tabela_faturas_atual]:
    df_faturas_atual = spark.table(tabela_faturas_atual)

    print(f"Registros na tabela atual: {df_faturas_atual.count():,}")

    comparar_schemas(
        df_faturas_atual,
        df_faturas_com_cadastro,
    )

    detalhe_atual = spark.sql(
        f"DESCRIBE DETAIL {tabela_faturas_atual}"
    )

    display(
        detalhe_atual.select(
            "format",
            "createdAt",
            "lastModified",
            "partitionColumns",
        )
    )

    if detalhe_atual.select("format").first()["format"] == "delta":
        display(
            spark.sql(
                f"DESCRIBE HISTORY {tabela_faturas_atual} LIMIT 1"
            ).select(
                "version",
                "timestamp",
                "operation",
                "operationParameters",
            )
        )
else:
    print("Não existe tabela atual de faturas nesse destino.")

for destino, dataframe in destinos_revisao:
    if existencia_destinos[destino]:
        print(f"Nome de revisão já ocupado: {destino}")
        comparar_schemas(spark.table(destino), dataframe)

Registros na tabela atual: 2,152,336
Nenhuma diferença de nomes ou tipos de colunas.


format,createdAt,lastModified,partitionColumns
delta,2026-10-23T13:33:40.431Z,2026-10-23T13:33:46.000Z,List()


version,timestamp,operation,operationParameters
1,2026-10-23T13:33:46.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""databricks.internal.autoUpgrades.delta.enableRowTracking"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""databricks.internal.autoUpgrades.delta.checkpointPolicy"":""v2"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""bcce3cec-723c-45e6-82fc-37b9e1a9c723"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f7cdc5d4-3b7c-4dc5-adbf-9f512ba37749"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-13ba1452-fbfc-48a6-83b3-fdb23a61ccba""}, statsOnLoad -> true)"


Nome de revisão já ocupado: `v-energy`.`silver`.`tb_faturas_revisao_v1`
Nenhuma diferença de nomes ou tipos de colunas.
Nome de revisão já ocupado: `v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`
Nenhuma diferença de nomes ou tipos de colunas.


In [0]:
A tabela principal já contém 2.152.336 faturas com correspondência
cadastral. As duas tabelas de revisão também existem.

Esta seção inspeciona os objetos atuais. A próxima seção confere
o conteúdo das revisões existentes, sem recriá-las.

## 12. Conferência das tabelas de revisão e retomada da execução

### Objetivo da seção

Esta seção verifica se as tabelas de revisão armazenadas no Databricks correspondem integralmente aos DataFrames que foram construídos e validados nas etapas anteriores.

As tabelas utilizadas são:

- `v-energy.silver.tb_faturas_revisao_v1`: faturas com correspondência cadastral.
- `v-energy.silver.tb_faturas_pendentes_cadastro_revisao_v1`: faturas sem correspondência cadastral.

Essas tabelas foram utilizadas como etapa intermediária da migração, permitindo conferir o resultado persistido antes da publicação nos destinos definitivos.

### Procedimento inicialmente proposto

A versão inicial desta seção continha células para:

1. Verificar se os nomes das tabelas estavam disponíveis.
2. Criar as duas tabelas de revisão.
3. Comparar os dados persistidos com os DataFrames esperados.
4. Apresentar a reconciliação.

Durante a execução e as retomadas, esse procedimento precisou ser ajustado porque as tabelas passaram a existir. Reexecutar a criação deixou de ser necessário.

### Erros encontrados e seus significados

#### 1. `UNSUPPORTED_OPERATION: errorifexists is not supported`

A tentativa de gravação utilizava `mode("errorifexists")`.

Esse alias não foi aceito pela implementação Spark Connect utilizada. A alternativa compatível naquela etapa foi `mode("error")`, cuja finalidade também era interromper a criação caso o destino já existisse.

Esse erro dizia respeito à chamada de gravação, não à qualidade dos dados.

#### 2. `ValueError: Destinos já existentes`

A verificação anterior à gravação encontrou as duas tabelas já existentes e interrompeu uma nova tentativa de criação.

A proteção funcionou conforme planejado. Entretanto, a existência das tabelas ainda não comprovava que seu conteúdo fosse igual ao esperado.

A próxima ação deveria ser ler e conferir os dados existentes, em vez de tentar criá-los novamente.

#### 3. `TABLE_OR_VIEW_ALREADY_EXISTS`

Esse erro apareceu quando uma célula antiga contendo `.write...saveAsTable(destino)` foi executada novamente.

A operação tentou criar uma tabela com um nome já ocupado. Isso mostrou que a célula de criação ainda estava no caminho de execução.

Não foi necessário apagar as tabelas ou sobrescrevê-las para resolver a situação. A solução foi substituir o procedimento de criação por leitura e conferência.

#### 4. Marcador de conferência ausente ou `False`

Na tentativa de avançar para a publicação, também apareceu uma mensagem informando que a conferência precisava estar concluída na sessão.

O marcador `REVISAO_PERSISTIDA_OK` é uma variável Python em memória. Ele pode estar ausente após um reinício de sessão ou ficar `False` quando uma célula de preparação é reexecutada.

Isso não significa que as tabelas persistidas desapareceram. Significa que a sessão ainda não registrou a conclusão da conferência.

O marcador não deve ser definido manualmente como `True`: a aprovação deve resultar da execução das verificações.

### Substituição das células antigas pelas células A e B

As células antigas desta seção foram retiradas da sequência executável e substituídas por duas células:

- **Célula A:** leitura das tabelas existentes e preparação da conferência.
- **Célula B:** comparação completa e reconciliação.

Foram substituídos os blocos antigos de criação, a verificação que bloqueava a execução por nomes já ocupados e os blocos de conferência que ficaram duplicados pela reorganização.

A remoção dessas células de código não significou apagar tabelas, registros ou decisões. O histórico das tentativas e suas justificativas permanece documentado neste texto.

### Célula A — carregar as tabelas existentes

A célula A:

1. Verifica se a candidata foi validada na sessão.
2. Associa cada tabela de revisão ao DataFrame esperado.
3. Confirma que as duas tabelas existem.
4. Consulta o histórico Delta e identifica uma versão para leitura.
5. Lê essa versão fixa e prepara os objetos utilizados pela célula B.

Na execução registrada, ambas as tabelas estavam na versão Delta 0.

Esta célula faz somente consultas. Ela não cria, substitui ou apaga tabelas.

### Célula B — conferir schema, conteúdo e reconciliação

A célula B compara cada tabela persistida com seu DataFrame esperado.

A conferência inclui:

- Nomes, ordem e tipos das colunas.
- Todas as colunas do conteúdo, incluindo originais e metadados.
- Comparação com `exceptAll` nas duas direções, preservando a multiplicidade dos registros.
- Quantidades e somas financeiras apresentadas na reconciliação.

A comparação nos dois sentidos permite detectar registros inesperados no destino e registros esperados que estejam ausentes.

Somente depois da aprovação das duas tabelas a célula define `REVISAO_PERSISTIDA_OK = True`.

Esta célula também faz somente consultas. Ela não publica os destinos definitivos.

### Resultado observado

| Tabela de revisão | Versão Delta | Faturas | Valor total |
| --- | ---: | ---: | ---: |
| `tb_faturas_revisao_v1` | 0 | 2.152.336 | R$ 666.482.773,76 |
| `tb_faturas_pendentes_cadastro_revisao_v1` | 0 | 32.285 | R$ 10.192.373,82 |
| **Total preservado** | — | **2.184.621** | **R$ 676.675.147,58** |

Os schemas e conteúdos completos das duas tabelas corresponderam aos DataFrames esperados.

### Papel desta seção no processo

A Seção 12 documenta a conferência das revisões da migração inicial. A publicação nos destinos definitivos ocorre separadamente, na Seção 13.

As tabelas de revisão são fotografias intermediárias. Sua conferência não significa que elas se atualizem automaticamente quando a Bronze recebe novos dados.

Para acrescentar ou revisar esta documentação, não é necessário executar as células A e B. Se for necessário repetir a conferência, os DataFrames e as validações anteriores precisam estar disponíveis na sessão.

In [0]:
REVISAO_PERSISTIDA_OK = False

if not globals().get("VALIDACOES_CANDIDATA_OK", False):
    raise ValueError(
        "A candidata precisa estar validada nesta sessão pela Seção 10."
    )

esperados_revisao = {
    f"{schema_silver}.`tb_faturas_revisao_v1`":
        df_faturas_com_cadastro,

    f"{schema_silver}.`tb_faturas_pendentes_cadastro_revisao_v1`":
        df_faturas_pendentes_cadastro,
}

faltantes = [
    tabela
    for tabela in esperados_revisao
    if not spark.catalog.tableExists(tabela)
]

if faltantes:
    raise ValueError(f"Tabelas não encontradas: {faltantes}")

revisoes_para_conferencia = {}

for tabela, esperado in esperados_revisao.items():
    historico = spark.sql(
        f"DESCRIBE HISTORY {tabela} LIMIT 1"
    ).select("version", "timestamp", "operation").first()

    versao = historico["version"]

    persistido = (
        spark.read
        .option("versionAsOf", versao)
        .table(tabela)
    )

    revisoes_para_conferencia[tabela] = {
        "versao": versao,
        "persistido": persistido,
        "esperado": esperado,
    }

    print(f"CARREGADA: {tabela}")
    print(f"Versão Delta: {versao}")
    print(f"Última operação: {historico['operation']}")

print("Somente leitura: nenhuma tabela foi criada ou alterada.")

CARREGADA: `v-energy`.`silver`.`tb_faturas_revisao_v1`
Versão Delta: 0
Última operação: CREATE TABLE AS SELECT
CARREGADA: `v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`
Versão Delta: 0
Última operação: CREATE TABLE AS SELECT
Somente leitura: nenhuma tabela foi criada ou alterada.


In [0]:
REVISAO_PERSISTIDA_OK = False

if set(revisoes_para_conferencia) != set(esperados_revisao):
    raise ValueError("Execute integralmente a célula A primeiro.")


def schema_para_conferencia(dataframe):
    return [
        (campo.name, campo.dataType.simpleString())
        for campo in dataframe.schema.fields
    ]


resumos_conferidos = []

for tabela, dados in revisoes_para_conferencia.items():
    persistido = dados["persistido"]
    esperado = dados["esperado"]

    if schema_para_conferencia(persistido) != schema_para_conferencia(esperado):
        raise ValueError(f"Schema divergente: {tabela}")

    colunas = esperado.columns
    atual = persistido.select(*colunas)
    referencia = esperado.select(*colunas)

    if atual.exceptAll(referencia).limit(1).count():
        raise ValueError(
            f"{tabela}: há registros persistidos diferentes do esperado."
        )

    if referencia.exceptAll(atual).limit(1).count():
        raise ValueError(
            f"{tabela}: há registros esperados ausentes na tabela."
        )

    print(f"OK: schema e conteúdo completo de {tabela}")

    resumo = (
        persistido
        .groupBy("situacao_referencia_cadastral")
        .agg(
            F.count("*").alias("faturas"),
            F.sum("valor_energia").alias("soma_energia"),
            F.sum("tributos").alias("soma_tributos"),
            F.sum("valor_total").alias("soma_total"),
        )
        .withColumn("tabela", F.lit(tabela))
        .withColumn("versao_delta", F.lit(dados["versao"]))
    )

    resumos_conferidos.append(resumo)

display(
    resumos_conferidos[0]
    .unionByName(resumos_conferidos[1])
    .select(
        "tabela",
        "versao_delta",
        "situacao_referencia_cadastral",
        "faturas",
        "soma_energia",
        "soma_tributos",
        "soma_total",
    )
)

REVISAO_PERSISTIDA_OK = True
print("OK: as duas tabelas existentes foram conferidas. Nenhuma gravação executada.")

OK: schema e conteúdo completo de `v-energy`.`silver`.`tb_faturas_revisao_v1`
OK: schema e conteúdo completo de `v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`


tabela,versao_delta,situacao_referencia_cadastral,faturas,soma_energia,soma_tributos,soma_total
`v-energy`.`silver`.`tb_faturas_revisao_v1`,0,CADASTRO_ENCONTRADO,2152336,499862082.73,166620691.03,666482773.76
`v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1`,0,PENDENTE_CADASTRO,32285,7644280.57,2548093.25,10192373.82


OK: as duas tabelas existentes foram conferidas. Nenhuma gravação executada.


## 13. Publicação definitiva e vínculo com a conferência

Esta seção documenta a publicação nos destinos:

- `silver.tb_faturas`: faturas com correspondência cadastral;
- `silver.tb_faturas_pendentes_cadastro`: faturas sem correspondência cadastral.

### Preparação e conferência das versões

A primeira célula de código prepara o plano de publicação, sem gravar dados.

O plano utiliza a versão 0 das duas tabelas de revisão. Antes de permitir
a escrita, o código exige que essas mesmas versões tenham sido conferidas
pelas células A e B da Seção 12.

Se faltar a conferência ou houver divergência de versão, a publicação
é bloqueada. Essa proteção corresponde à correção da issue #28.

### Confirmação explícita da publicação

A segunda célula de código contém a configuração:

`CONFIRMAR_PUBLICACAO = False`

Esse é o padrão para revisão do notebook.

Com esse valor, a execução é interrompida antes de qualquer gravação,
inclusive quando o notebook é executado com Run All. A mensagem
“Publicação bloqueada” é esperada e demonstra que a proteção funcionou.

A confirmação somente deve ser alterada para True quando houver intenção
de publicar novamente os dois destinos, após a conferência do plano.
Depois dessa operação, o código deve ser salvo novamente com False.

Também são verificadas alterações nos destinos entre a preparação
do plano e a escrita.

### Conferência após uma publicação autorizada

Quando a publicação é autorizada, as células seguintes conferem schema
e conteúdo dos destinos nas versões Delta registradas pela publicação,
utilizando comparação completa com multiplicidade.

As duas gravações são transações separadas. Uma falha entre elas pode
deixar uma publicação parcial, que deve ser investigada antes de continuar.

### Evidências históricas

Os outputs de publicação e de conferência final preservados nas células
posteriores registram uma execução anterior.

Na rodada interrompida por CONFIRMAR_PUBLICACAO = False, essas células
posteriores não foram executadas. Seus outputs não comprovam uma nova
publicação.

Esta seção representa a migração inicial baseada nas tabelas de revisão.
Ela não constitui uma rotina mensal de atualização a partir da Bronze.

In [0]:
PUBLICACAO_FATURAS_OK = False
plano_publicacao = []


def validar_fontes_conferidas(plano):
    if not globals().get("VALIDACOES_CANDIDATA_OK", False):
        raise ValueError("Conclua as validações da Seção 10 nesta sessão.")

    if not globals().get("REVISAO_PERSISTIDA_OK", False):
        raise ValueError("Execute as células A e B da Seção 12 nesta sessão.")

    conferencias = globals().get("revisoes_para_conferencia")

    if not isinstance(conferencias, dict):
        raise ValueError("Registro de conferência ausente. Execute A e B da Seção 12.")

    destinos_por_fonte = {
        f"{schema_silver}.`tb_faturas_pendentes_cadastro_revisao_v1`":
            f"{schema_silver}.`tb_faturas_pendentes_cadastro`",
        f"{schema_silver}.`tb_faturas_revisao_v1`":
            f"{schema_silver}.`tb_faturas`",
    }

    if len(plano) != 2 or {item["fonte"] for item in plano} != set(destinos_por_fonte):
        raise ValueError("O plano precisa conter exatamente as duas fontes da migração.")

    for item in plano:
        fonte = item["fonte"]
        dados = conferencias.get(fonte)

        if item["destino"] != destinos_por_fonte[fonte] or item["versao_fonte"] != 0:
            raise ValueError("O plano diverge dos destinos e da versão 0 aprovados.")

        if not isinstance(dados, dict) or dados.get("persistido") is None:
            raise ValueError(f"Sem registro de conferência para {fonte}. Execute A e B.")

        if dados.get("versao") != item["versao_fonte"]:
            raise ValueError(
                f"Versão divergente em {fonte}: "
                f"conferida={dados.get('versao')}, "
                f"planejada={item['versao_fonte']}. "
                "Nenhuma gravação foi iniciada por esta célula. "
                "Não altere a versão para contornar o bloqueio."
            )


def obter_estado_delta(tabela):
    if not spark.catalog.tableExists(tabela):
        return None

    detalhe = spark.sql(f"DESCRIBE DETAIL {tabela}").first()

    if detalhe["format"] != "delta":
        raise ValueError(f"O destino existente não é Delta: {tabela}")

    versao = spark.sql(
        f"DESCRIBE HISTORY {tabela} LIMIT 1"
    ).select("version").first()["version"]

    return {"id": detalhe["id"], "versao": versao}


plano_em_preparacao = [
    {
        "fonte": f"{schema_silver}.`tb_faturas_pendentes_cadastro_revisao_v1`",
        "versao_fonte": 0,
        "destino": f"{schema_silver}.`tb_faturas_pendentes_cadastro`",
    },
    {
        "fonte": f"{schema_silver}.`tb_faturas_revisao_v1`",
        "versao_fonte": 0,
        "destino": f"{schema_silver}.`tb_faturas`",
    },
]

# Conferir ambas as origens antes de disponibilizar o plano para gravação.
validar_fontes_conferidas(plano_em_preparacao)

for item in plano_em_preparacao:
    # Reutilizar o snapshot que foi efetivamente comparado na Seção 12.
    item["referencia"] = revisoes_para_conferencia[item["fonte"]]["persistido"]
    item["estado_anterior"] = obter_estado_delta(item["destino"])

    print(f"FONTE CONFERIDA: {item['fonte']} — versão {item['versao_fonte']}")
    print(f"DESTINO: {item['destino']}")
    print(f"Registros previstos: {item['referencia'].count():,}")

plano_publicacao = plano_em_preparacao
print("OK: plano vinculado às versões conferidas. Nenhuma tabela foi gravada.")


FONTE CONFERIDA: `v-energy`.`silver`.`tb_faturas_pendentes_cadastro_revisao_v1` — versão 0
DESTINO: `v-energy`.`silver`.`tb_faturas_pendentes_cadastro`
Registros previstos: 32,285
FONTE CONFERIDA: `v-energy`.`silver`.`tb_faturas_revisao_v1` — versão 0
DESTINO: `v-energy`.`silver`.`tb_faturas`
Registros previstos: 2,152,336
OK: plano vinculado às versões conferidas. Nenhuma tabela foi gravada.


In [0]:
# Padrão seguro para revisão e Run All.
# Altere para True somente se quiser publicar novamente os dois destinos.
CONFIRMAR_PUBLICACAO = False

PUBLICACAO_FATURAS_OK = False
publicacoes_realizadas = {}

if CONFIRMAR_PUBLICACAO is not True:
    raise RuntimeError(
        "Publicação bloqueada: CONFIRMAR_PUBLICACAO está False. "
        "Nenhuma tabela foi gravada por esta célula. "
        "Para uma publicação intencional, revise o plano e altere "
        "a confirmação para True nesta célula."
    )

# Preserva a proteção da issue #28.
# A versão planejada deve corresponder à versão efetivamente conferida.
validar_fontes_conferidas(plano_publicacao)

# Conferir os dois destinos antes da primeira escrita.
for item in plano_publicacao:
    if obter_estado_delta(item["destino"]) != item["estado_anterior"]:
        raise ValueError(
            f"O destino mudou desde a preparação: {item['destino']}. "
            "Nenhuma gravação foi iniciada por esta célula."
        )

for item in plano_publicacao:
    destino = item["destino"]

    # Repetir a conferência imediatamente antes de cada escrita.
    if obter_estado_delta(destino) != item["estado_anterior"]:
        raise ValueError(
            f"O destino mudou antes da gravação: {destino}. "
            "Investigue eventual publicação parcial antes de continuar."
        )

    spark.sql(f"""
        CREATE OR REPLACE TABLE {destino}
        USING DELTA
        AS
        SELECT *
        FROM {item["fonte"]}
        VERSION AS OF {item["versao_fonte"]}
    """)

    historico = spark.sql(
        f"DESCRIBE HISTORY {destino} LIMIT 1"
    ).select(
        "version",
        "timestamp",
        "operation",
    ).first()

    versao_publicada = historico["version"]

    publicacoes_realizadas[destino] = {
        "versao": versao_publicada,
        "referencia": item["referencia"],
        "persistido": (
            spark.read
            .option("versionAsOf", versao_publicada)
            .table(destino)
        ),
    }

    print(f"PUBLICADA: {destino}")
    print(f"Versão Delta: {versao_publicada}")
    print(f"Operação: {historico['operation']}")

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-4972414344055023>, line 9
      6 publicacoes_realizadas = {}
      8 if CONFIRMAR_PUBLICACAO is not True:
----> 9     raise RuntimeError(
     10         "Publicação bloqueada: CONFIRMAR_PUBLICACAO está False. "
     11         "Nenhuma tabela foi gravada por esta célula. "
     12         "Para uma publicação intencional, revise o plano e altere "
     13         "a confirmação para True nesta célula."
     14     )
     16 # Preserva a proteção da issue #28.
     17 # A versão planejada deve corresponder à versão efetivamente conferida.
     18 validar_fontes_conferidas(plano_publicacao)

RuntimeError: Publicação bloqueada: CONFIRMAR_PUBLICACAO está False. Nenhuma tabela foi gravada por esta célula. Para uma publicação intencional, revise o plano e altere a confirmação para True nesta célula.

In [0]:
destinos_esperados = {
    item["destino"] for item in plano_publicacao
}

if set(publicacoes_realizadas) != destinos_esperados:
    raise ValueError("A publicação dos dois destinos não foi concluída.")


def contrato_publicacao(dataframe):
    return [
        (campo.name, campo.dataType.simpleString())
        for campo in dataframe.schema.fields
    ]


destinos_validados = set()

for destino, dados in publicacoes_realizadas.items():
    persistido = dados["persistido"]
    referencia = dados["referencia"]

    if contrato_publicacao(persistido) != contrato_publicacao(referencia):
        raise ValueError(f"Schema publicado divergente: {destino}")

    conferir_mesmos_registros(
        persistido,
        referencia,
        referencia.columns,
        f"Conteúdo definitivo de {destino}",
    )

    destinos_validados.add(destino)

print("OK: schema e conteúdo dos dois destinos definitivos.")

OK: Conteúdo definitivo de `v-energy`.`silver`.`tb_faturas_pendentes_cadastro`.
OK: Conteúdo definitivo de `v-energy`.`silver`.`tb_faturas`.
OK: schema e conteúdo dos dois destinos definitivos.


In [0]:
if destinos_validados != destinos_esperados:
    raise ValueError("Falta validar algum destino definitivo.")

resumos_publicacao = []

for destino, dados in publicacoes_realizadas.items():
    resumo = (
        dados["persistido"]
        .groupBy("situacao_referencia_cadastral")
        .agg(
            F.count("*").alias("faturas"),
            F.sum("valor_energia").alias("soma_energia"),
            F.sum("tributos").alias("soma_tributos"),
            F.sum("valor_total").alias("soma_total"),
        )
        .withColumn("tabela", F.lit(destino))
        .withColumn("versao_delta", F.lit(dados["versao"]))
    )

    resumos_publicacao.append(resumo)

display(
    resumos_publicacao[0]
    .unionByName(resumos_publicacao[1])
    .select(
        "tabela",
        "versao_delta",
        "situacao_referencia_cadastral",
        "faturas",
        "soma_energia",
        "soma_tributos",
        "soma_total",
    )
)

PUBLICACAO_FATURAS_OK = True
print("OK: publicação definitiva concluída e conferida.")

tabela,versao_delta,situacao_referencia_cadastral,faturas,soma_energia,soma_tributos,soma_total
`v-energy`.`silver`.`tb_faturas_pendentes_cadastro`,0,PENDENTE_CADASTRO,32285,7644280.57,2548093.25,10192373.82
`v-energy`.`silver`.`tb_faturas`,1,CADASTRO_ENCONTRADO,2152336,499862082.73,166620691.03,666482773.76


OK: publicação definitiva concluída e conferida.
